In [ ]:
# GOOGLE VERIFIABLE QUANTUM CIRCUIT COMPILATION
from google.colab import drive
from pathlib import Path
import zipfile
import json
import os
import shutil
import re
from google.colab import drive
import csv
import ast
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
BASE_DIR = Path(
    "/content/drive/MyDrive/Quantum_Competition"
)

ZIP_PATH = BASE_DIR / (
    "google-verifiable-quantum-circuit-compilation.zip"
)

DATA_DIR = BASE_DIR / "Extracted"

MARKER = DATA_DIR / ".extraction_complete.json"

BASE_DIR.mkdir(parents=True, exist_ok=True)

def extraction_is_valid():

    if not MARKER.exists():
        return False

    try:
        manifest = json.loads(MARKER.read_text())

        for item in manifest["files"]:
            path = DATA_DIR / item["name"]

            if not path.is_file():
                return False

            if path.stat().st_size != item["size"]:
                return False

        return True

    except Exception:
        return False


if extraction_is_valid():

    print("EXISTING VERIFIED DATASET FOUND")
    print("Skipping extraction.")

else:

    print("No verified extraction found.")

    if not ZIP_PATH.exists():
        raise FileNotFoundError(
            f"Fresh Kaggle ZIP required at: {ZIP_PATH}"
        )


    print("\nChecking ZIP archive...")

    if not zipfile.is_zipfile(ZIP_PATH):
        raise ValueError(
            "Invalid ZIP. Download a fresh archive from Kaggle."
        )

    with zipfile.ZipFile(ZIP_PATH, "r") as z:

        members = z.infolist()


        for item in members:
            target = (DATA_DIR / item.filename).resolve()

            if not target.is_relative_to(DATA_DIR.resolve()):
                raise ValueError(
                    f"Unsafe archive path: {item.filename}"
                )

        print("Archive entries:", len(members))

        print("Checking file integrity...")

        bad_file = z.testzip()

        if bad_file:
            raise ValueError(
                f"Corrupted archive member: {bad_file}. "
                "Download the ZIP again from Kaggle."
            )

        print("ZIP INTEGRITY VERIFIED")


        TEMP_DIR = BASE_DIR / "Extracted_TEMP"

        if TEMP_DIR.exists():
            shutil.rmtree(TEMP_DIR)

        TEMP_DIR.mkdir(parents=True)

        print("\nExtracting dataset...")

        z.extractall(TEMP_DIR)


        manifest = {
            "archive": ZIP_PATH.name,
            "files": [
                {
                    "name": str(p.relative_to(TEMP_DIR)),
                    "size": p.stat().st_size
                }
                for p in TEMP_DIR.rglob("*")
                if p.is_file()
            ]
        }

        (TEMP_DIR / ".extraction_complete.json").write_text(
            json.dumps(manifest, indent=2)
        )


        if DATA_DIR.exists():
            shutil.rmtree(DATA_DIR)

        TEMP_DIR.rename(DATA_DIR)

        print("EXTRACTION VERIFIED AND SAVED")

DATA_ROOT = DATA_DIR

print("\n" + "=" * 60)
print("QUANTUM COMPETITION DATASET READY")
print("=" * 60)

print("Dataset location:", DATA_ROOT)

print("\nFOLDER STRUCTURE\n")

for path in sorted(DATA_ROOT.rglob("*")):

    if path.name == ".extraction_complete.json":
        continue

    relative = path.relative_to(DATA_ROOT)

    if path.is_dir():
        print("[DIR]", relative)

    else:
        print(
            "[FILE]", relative,
            f"({path.stat().st_size / 1024:.2f} KB)"
        )

all_files = [
    p for p in DATA_ROOT.rglob("*")
    if p.is_file()
    and p.name != ".extraction_complete.json"
]

print("\nTOTAL FILES:", len(all_files))
print("TOTAL SIZE (MB):", round(
    sum(p.stat().st_size for p in all_files) / 1024**2, 2
))

print("\nDATASET LOADED SUCCESSFULLY")


EXISTING VERIFIED DATASET FOUND
Skipping extraction.

QUANTUM COMPETITION DATASET READY
Dataset location: /content/drive/MyDrive/Quantum_Competition/Extracted

FOLDER STRUCTURE

[DIR] kickmix
[FILE] kickmix/LICENSE.txt (11.09 KB)
[FILE] kickmix/kickmix.so (3144.91 KB)
[DIR] sample_submission
[DIR] sample_submission/dfthcsos
[FILE] sample_submission/dfthcsos/angle_k_0.kmb (1632.75 KB)
[FILE] sample_submission/dfthcsos/angle_k_1.kmb (1623.92 KB)
[FILE] sample_submission/dfthcsos/angle_k_2.kmb (1626.78 KB)
[FILE] sample_submission/dfthcsos/angle_k_3.kmb (1627.74 KB)
[FILE] sample_submission/dfthcsos/angle_k_4.kmb (1908.28 KB)
[FILE] sample_submission/dfthcsos/angle_k_5.kmb (1902.91 KB)
[FILE] sample_submission/dfthcsos/angle_k_6.kmb (1902.32 KB)
[FILE] sample_submission/dfthcsos/angle_k_7.kmb (1922.39 KB)
[FILE] sample_submission/dfthcsos/angle_k_8.kmb (1911.25 KB)
[FILE] sample_submission/dfthcsos/angle_k_9.kmb (1908.04 KB)
[FILE] sample_submission/dfthcsos/submission.npz (4431.09 KB)
[F

In [ ]:
drive.mount('/content/drive')

DATA_ROOT = Path(
    "/content/drive/MyDrive/Quantum_Competition/Extracted"
)

assert (DATA_ROOT / "verifiers/dqi/verify_submission_dqi.py").exists()
assert (DATA_ROOT / "verifiers/invsqrt/verify_submission_invsqrt.py").exists()
assert (DATA_ROOT / "verifiers/dfthcsos/verify.py").exists()
assert (DATA_ROOT / "sample_submission/dqi.kmb").exists()

print("ALL COMPETITION FILES ACCESSIBLE")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ALL COMPETITION FILES ACCESSIBLE


In [ ]:
ROOT = Path(
    "/content/drive/MyDrive/Quantum_Competition/Extracted"
)

print("COMPETITION ENVIRONMENT CHECK\n")

tasks = {
    "INVSQRT": ROOT / "verifiers/invsqrt/verify_submission_invsqrt.py",
    "DQI": ROOT / "verifiers/dqi/verify_submission_dqi.py",
    "DFTHCSOS": ROOT / "verifiers/dfthcsos/verify.py"
}

for name, path in tasks.items():
    print(f"\n{'='*45}")
    print(name)
    print("Verifier exists:", path.exists())

    if path.exists():
        lines = path.read_text(errors="replace").splitlines()

        relevant = [
            (i, line.strip())
            for i, line in enumerate(lines)
            if any(term in line.lower() for term in [
                "add_argument",
                "argparse",
                "submission",
                "log2_delta",
                "threshold",
                "def main",
                "if __name__"
            ])
        ]

        for i, line in relevant[:18]:
            print(f"L{i+1}: {line[:180]}")

# Scoring documentation
readme = ROOT / "verifiers/dfthcsos/SCORING_README.md"

print("\n\nSCORING DOCUMENTATION")

if readme.exists():
    lines = readme.read_text(errors="replace").splitlines()

    for i, line in enumerate(lines):
        if line.strip().startswith("#"):
            print(f"L{i+1}: {line[:150]}")

print("\nENVIRONMENT INSPECTION COMPLETE")


COMPETITION ENVIRONMENT CHECK


INVSQRT
Verifier exists: True
L3: A submission is a kickmix circuit implementing, for every `x` in `[1, max_x]`,
L44: python invsqrt/verify_submission_invsqrt.py submission.kmb [--shots-per-x N] [--salt STR] [--seed N]
L49: import argparse
L347: def main() -> None:
L348: parser = argparse.ArgumentParser(description=__doc__,
L349: formatter_class=argparse.RawDescriptionHelpFormatter)
L350: parser.add_argument('circuit_file', help='Path to a submitted .kmb or .kmx circuit file.')
L351: parser.add_argument('--format', choices=('auto', 'kmb', 'kmx'), default='auto',
L353: parser.add_argument('--n-p', type=int, default=N_P,
L355: parser.add_argument('--beta', type=Fraction, default=Fraction(1),
L357: parser.add_argument('--precision-bits', type=int, default=24,
L359: parser.add_argument('--shots-per-x', type=int, default=DEFAULT_SHOTS_PER_X,
L361: parser.add_argument('--eta', type=int, default=ETA,
L363: parser.add_argument('--width', type=int, default=None,


In [ ]:
ROOT = Path("/content/drive/MyDrive/Quantum_Competition/Extracted")

verifier = ROOT / "verifiers/dqi/verify_submission_dqi.py"
sample = ROOT / "sample_submission/dqi.kmb"

lines = verifier.read_text(errors="replace").splitlines()

sections = [
    (1, 75),       # Rules and defaults
    (130, 180),    # Statistical error calculation
    (250, 370),    # Validation and verification
    (370, 460)     # CLI and output
]

print("DQI VERIFIER INSPECTION\n")

for start, end in sections:
    if start > len(lines):
        continue
    print(f"\n{'='*55}")
    print(f"LINES {start}-{min(end, len(lines))}")
    for i in range(start - 1, min(end, len(lines))):
        print(f"{i+1}: {lines[i]}")

print("\nSAMPLE SUBMISSION")
print("Exists:", sample.exists())
if sample.exists():
    print("Size (MB):", round(sample.stat().st_size / (1024**2), 2))

print("\nINSPECTION COMPLETE")


DQI VERIFIER INSPECTION


LINES 1-75
1: """Verifier for Benchmark II: DQI Reed-Solomon MBUC syndrome decoder (arXiv:2408.08292, arXiv:2510.10967).
2: 
3: For parameters `(m, n, b)` (`m` codeword positions, `n` syndromes correcting up to
4: `l = n // 2` errors over `GF(2^b)` using `km.GF2Field(b)` with primitive element
5: `alpha = km.GF2Field(b).primitive_element`):
6: 
7: Registers (little-endian, width `b`, pairwise disjoint, no other registers allowed):
8:     S0..S{n-1}  Quantum syndrome registers (`km.q`). Hold `S_i = sum_j e_j alpha^{(i+1) j}`.
9:     E0..E{m-1}  Classical MBUC outcome registers (`km.b`). Hold `M_0..M_{m-1}` on input.
10: 
11: Contract: on each shot, `S0..S{n-1}` must be restored, all other qubits must return to |0>,
12: and the global phase must be multiplied by `(-1) ** sum_j popcount(e_j & M_j)`. `Z_POW` gates
13: are not allowed. With `k = value_errors + phase_errors` failed shots out of `N = num_shots`,
14: the one-sided Clopper-Pearson upper bound `eps_plus

In [ ]:
ROOT = Path("/content/drive/MyDrive/Quantum_Competition/Extracted")
p = ROOT / "verifiers/dqi/verify_submission_dqi.py"
lines = p.read_text(errors="replace").splitlines()

print("STATISTICAL BOUND")
for i in range(130, min(180, len(lines))):
    print(f"{i+1}: {lines[i]}")

print("\nVERIFIER COMMAND OPTIONS")
for i in range(443, min(510, len(lines))):
    print(f"{i+1}: {lines[i]}")

print("\nSAMPLE FILE")
sample = ROOT / "sample_submission/dqi.kmb"
print("Available:", sample.exists())
print("Size MB:", round(sample.stat().st_size / (1024**2), 2))


STATISTICAL BOUND
131:     """Repetition factor `R(eps) = ln(2) / -ln(1/2 + 2 * sqrt(eps))` for `0 <= eps < 1/16`."""
132:     if not 0.0 <= eps < 1 / 16:
133:         raise ValueError(f'Effective error rate eps must be in [0, 1/16), got {eps}.')
134:     if eps == 0.0:
135:         return 1.0
136:     return math.log(2.0) / -math.log(0.5 + 2.0 * math.sqrt(eps))
137: 
138: 
139: def clopper_pearson_upper_bound(
140:     k: int, n: int, log2_delta: float = DEFAULT_LOG2_DELTA
141: ) -> float:
142:     """One-sided Clopper-Pearson upper bound `eps_plus` at confidence `1 - 2^log2_delta`."""
143:     if n <= 0 or not (0 <= k <= n) or not (math.isfinite(log2_delta) and log2_delta <= 0.0):
144:         raise ValueError(
145:             f'Expected 0 <= k <= n, n > 0, and finite log2_delta <= 0, '
146:             f'got k={k}, n={n}, log2_delta={log2_delta}.'
147:         )
148:     if k == n or log2_delta == 0.0:
149:         return 1.0 if k == n else k / n
150:     log_delta = log2_delta * m

In [ ]:
sys.path.insert(0, str(ROOT / "kickmix"))

import kickmix as km

sample = ROOT / "sample_submission/dqi.kmb"

print("Loading official DQI sample...")
circuit = km.Circuit.from_file(str(sample), format="kmb")

print("\nOFFICIAL SAMPLE CIRCUIT")
print("Qubits:", circuit.num_qubits)
print("Classical bits:", circuit.num_bits)
print("Total operations:", len(circuit))

stats = circuit.operation_statistics()

print("\nOPERATION STATISTICS")
print(stats)

print("\nREACTION DEPTH")
print(circuit.reaction_depth())

print("\nBASELINE INSPECTION COMPLETE")


Loading official DQI sample...

OFFICIAL SAMPLE CIRCUIT
Qubits: 1893
Classical bits: 49154
Total operations: 86051014

OPERATION STATISTICS
<km.OperationStatistics num_instructions=86051014 num_toffoli_like=5725461 num_t_like=0>

REACTION DEPTH
170757

BASELINE INSPECTION COMPLETE


In [ ]:
EXPERIMENTS = BASE / "Experiments"
FINAL = BASE / "Final_Submission"

folders = [
    EXPERIMENTS / "DQI" / "circuits",
    EXPERIMENTS / "INVSQRT" / "circuits",
    EXPERIMENTS / "DFTHCSOS" / "circuits",
    FINAL / "dfthcsos"
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

headers = [
    "experiment_id", "benchmark", "version",
    "status", "qubits", "total_operations",
    "toffoli_count", "reaction_depth",
    "error_rate", "notes"
]

for task in ["DQI", "INVSQRT", "DFTHCSOS"]:
    log = EXPERIMENTS / task / "experiment_log.csv"

    if not log.exists():
        with open(log, "w", newline="") as f:
            csv.writer(f).writerow(headers)

# Recording official DQI baseline
log = EXPERIMENTS / "DQI" / "experiment_log.csv"

with open(log, "a", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "DQI_BASELINE",
        "DQI",
        "official_sample",
        "INSPECTED",
        1893,
        86051014,
        5725461,
        170757,
        "",
        "Official competition sample; circuit not yet verified"
    ])

baseline = {
    "benchmark": "DQI",
    "qubits": 1893,
    "classical_bits": 49154,
    "total_operations": 86051014,
    "toffoli_like": 5725461,
    "reaction_depth": 170757,
    "verification": "NOT_RUN"
}

with open(EXPERIMENTS / "DQI" / "baseline.json", "w") as f:
    json.dump(baseline, f, indent=4)

print("PERSISTENT TRACKER CREATED")
print("Location:", BASE)
print("DQI baseline saved.")
print("CSV experiment logs ready.")


PERSISTENT TRACKER CREATED
Location: /content/drive/MyDrive/Quantum_Competition
DQI baseline saved.
CSV experiment logs ready.


In [ ]:
ROOT = Path("/content/drive/MyDrive/Quantum_Competition/Extracted")

p = ROOT / "verifiers/dqi/dqi_reference_circuit.py"
lines = p.read_text(errors="replace").splitlines()

print("DQI REFERENCE IMPLEMENTATION")
print("Total lines:", len(lines))

for i, line in enumerate(lines[:180]):
    if (
        line.startswith("def ")
        or line.startswith("class ")
        or line.startswith("import ")
        or line.startswith("from ")
        or "Circuit" in line
        or "Toffoli" in line
    ):
        print(f"L{i+1}: {line[:180]}")

print("\nREFERENCE INSPECTION COMPLETE")


DQI REFERENCE IMPLEMENTATION
Total lines: 647
L25: from __future__ import annotations
L27: import argparse
L28: from typing import Sequence
L30: import kickmix as km
L36: def _active(cursor: km.UnaryIterationCursor, address_value: int, limit: int):
L50: def _xor_threshold(builder: km.CircuitBuilder, cursor: km.UnaryIterationCursor,
L64: class ProosAndZalkaDecoder:
L73:     def build_syndrome_decoding_mbuc(self, builder: km.CircuitBuilder, *,
L80:             builder: Circuit builder receiving the instructions.
L140:     def _initial_shift_loop(self, builder: km.CircuitBuilder,
L166:     def _synchronized_eea(self, builder: km.CircuitBuilder,

REFERENCE INSPECTION COMPLETE


In [ ]:
ROOT = Path("/content/drive/MyDrive/Quantum_Competition/Extracted")
p = ROOT / "verifiers/dqi/dqi_reference_circuit.py"

source = p.read_text(errors="replace")
tree = ast.parse(source)

print("DQI DECODER FUNCTION MAP\n")

for node in ast.walk(tree):
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        args = [a.arg for a in node.args.args]
        print(
            f"Line {node.lineno:3} | "
            f"{node.name}({', '.join(args)})"
        )

print("\nFUNCTION MAP COMPLETE")


DQI DECODER FUNCTION MAP

Line  36 | _active(cursor, address_value, limit)
Line  50 | _xor_threshold(builder, cursor, limit, target, control, value)
Line 613 | default_n_iters(n_syndromes)
Line 618 | build_reference_circuit(m, n, degree)
Line 628 | main()
Line  67 | __init__(self, n_syndromes, field)
Line  73 | build_syndrome_decoding_mbuc(self, builder)
Line 140 | _initial_shift_loop(self, builder, reg_a, n_A)
Line 166 | _synchronized_eea(self, builder, reg_a, reg_b, n_A, n_B, n_a, n_b, n_q, flag, counter, halting_counter)
Line 218 | _counter_controls(builder, counter, is_c3, is_hc_zero)
Line 230 | _del_counter_controls(builder, counter, is_c3, is_hc_zero, ctrl)
Line 240 | _update_counter(builder, counter, flag)
Line 250 | _update_flag(self, builder, reg_b, n_A, n_B, n_a, n_b, n_q, flag, ctrl)
Line 291 | _multiply_and_add_or_sub(self, builder, reg_a, reg_b, n_A, n_a, n_q, ctrl, it)
Line 378 | _shift_b_array_right(self, builder, reg_b, n_B, n_b, flag, ctrl, it)
Line 440 | _update_metad

In [ ]:
ROOT = Path("/content/drive/MyDrive/Quantum_Competition/Extracted")
p = ROOT / "verifiers/dqi/dqi_reference_circuit.py"
lines = p.read_text(errors="replace").splitlines()

ranges = [
    (166, 217),  # Synchronized EEA
    (291, 377),  # Multiply and add/subtract
    (378, 439),  # Shift operations
    (440, 508),  # Metadata, swaps and halting
]

for start, end in ranges:
    print(f"\n{'='*50}")
    print(f"LINES {start}-{end}")
    for i in range(start - 1, min(end, len(lines))):
        print(f"{i+1}: {lines[i]}")

print("\nEEA INSPECTION COMPLETE")



LINES 166-217
166:     def _synchronized_eea(self, builder: km.CircuitBuilder,
167:                           reg_a: Sequence[km.array], reg_b: Sequence[km.array],
168:                           n_A: km.array, n_B: km.array, n_a: km.array,
169:                           n_b: km.array, n_q: km.array, flag,
170:                           counter: km.array, halting_counter: km.array, *,
171:                           n_iters: int, is_adjoint: bool) -> None:
172:         """Runs `n_iters` steps of the reversible synchronized EEA."""
173:         eea_step = builder.mark('eea_step')
174:         for it in (range(n_iters - 1, -1, -1) if is_adjoint else range(n_iters)):
175:             with eea_step:
176:                 if is_adjoint:
177:                     self._update_counter(builder, counter, flag, is_adjoint=True)
178: 
179:                 is_c3 = builder.alloc_qubits(1)[0]
180:                 builder.init_and(counter[0], counter[1], is_c3)
181:                 if is_adjoint:
182:  

In [ ]:
ROOT = Path("/content/drive/MyDrive/Quantum_Competition/Extracted")
p = ROOT / "verifiers/dqi/dqi_reference_circuit.py"
lines = p.read_text(errors="replace").splitlines()

print("DECODER CONSTRUCTION PARAMETERS\n")

for start, end in [(64, 140), (509, 612), (613, 647)]:
    print(f"\n{'='*45}")
    for i in range(start - 1, min(end, len(lines))):
        print(f"{i+1}: {lines[i]}")

print("\nPARAMETER INSPECTION COMPLETE")


DECODER CONSTRUCTION PARAMETERS


64: class ProosAndZalkaDecoder:
65:     """Builds the reversible Reed-Solomon MBUC decoder circuit (arXiv:2510.10967)."""
66: 
67:     def __init__(self, n_syndromes: int, field: km.GF2Field):
68:         if n_syndromes < 2:
69:             raise ValueError(f'n_syndromes must be at least 2, got {n_syndromes}.')
70:         self.n_syndromes = n_syndromes
71:         self.field = field
72: 
73:     def build_syndrome_decoding_mbuc(self, builder: km.CircuitBuilder, *,
74:                                      syndromes: Sequence[km.array],
75:                                      error_mbuc: Sequence[km.array],
76:                                      n_iters: int) -> None:
77:         """Appends the syndrome-decoding MBUC phase circuit to `builder`.
78: 
79:         Args:
80:             builder: Circuit builder receiving the instructions.
81:             syndromes: `n_syndromes` quantum registers of width `field.degree`.
82:                 Restored to t

In [ ]:
ROOT = Path("/content/drive/MyDrive/Quantum_Competition/Extracted")

p = ROOT / "verifiers/dqi/verify_submission_dqi.py"
lines = p.read_text(errors="replace").splitlines()

for i, line in enumerate(lines):
    if any(term in line.lower() for term in [
        "build_reference_circuit",
        "default_n_iters",
        "n_iters",
        "reference_circuit",
        "n_shots"
    ]):
        print(f"{i+1}: {line[:220]}")

In [ ]:
ROOT = Path("/content/drive/MyDrive/Quantum_Competition/Extracted")
p = ROOT / "verifiers/dqi/verify_submission_dqi.py"
lines = p.read_text(errors="replace").splitlines()

for start, end in [(320, 390), (440, 505)]:
    print(f"\n{'='*50}\nLINES {start}-{end}\n")
    for i in range(start - 1, min(end, len(lines))):
        print(f"{i+1}: {lines[i]}")

print("\nVERIFIER INSPECTION COMPLETE")


LINES 320-390

320:     if workers <= 1 or 'fork' not in mp.get_all_start_methods():
321:         return [fn(t) for t in tasks]
322:     _WORKER_FN = fn
323:     try:
324:         with concurrent.futures.ProcessPoolExecutor(max_workers=workers, mp_context=mp.get_context('fork')) as pool:
325:             try:
326:                 return list(pool.map(_call_worker, tasks))
327:             except Exception:
328:                 pool.shutdown(wait=True, cancel_futures=True)
329:                 raise
330:     finally:
331:         _WORKER_FN = None
332: 
333: 
334: def verify(
335:     circuit: km.Circuit,
336:     spec: ProblemSpecDQI,
337:     *,
338:     seed: int | None = None,
339:     salt: str = '',
340:     batch_size: int = DEFAULT_BATCH_SIZE,
341:     log2_delta: float = DEFAULT_LOG2_DELTA,
342:     max_workers: int | None = None,
343:     shots: tuple[np.ndarray, np.ndarray, np.ndarray] | None = None,
344: ) -> dict[str, Any]:
345:     """Verifies `circuit` on `spec.num_shots

In [ ]:
import kickmix as km

print("KickMix optimisation-related methods:\n")

for name in dir(km):
    if any(word in name.lower() for word in [
        "optim", "simpl", "synth", "compile", "decomp"
    ]):
        print(name)

print("\nCircuit methods:\n")

for name in dir(km.Circuit):
    if any(word in name.lower() for word in [
        "optim", "simpl", "synth", "compile", "decomp"
    ]):
        print(name)

KickMix optimisation-related methods:


Circuit methods:



In [ ]:
import kickmix as km

print("CircuitBuilder methods:\n")

for name in dir(km.CircuitBuilder):
    if not name.startswith("_"):
        print(name)

print("\nKickMix module location:")
print(km.__file__)

CircuitBuilder methods:

alloc_bits
alloc_qubits
bit_store
ccx
ccz
create_classical_register
create_quantum_register
cswap
cx
cz
debug_print
del_and
del_gf2_div
del_gf2_div_with_scaffold
del_gf2_inverse
del_gf2_inverse_with_scaffold
del_gf2_mul
del_lookup
finish_circuit
flip_if_equal
flip_if_greater_than
flip_if_less_than
free
gf2_iadd
gf2_idiv
gf2_imul
gf2_phase_by_product
hmr
iadd
info_tag
init_and
init_gf2_div
init_gf2_div_with_scaffold
init_gf2_inverse
init_gf2_inverse_with_scaffold
init_gf2_mul
init_lookup
isub
ixor_gf2_div
ixor_gf2_mul
left_rotate
mark
max_qubits
multi_controlled_x
neg
neg_if
num_allocated_qubits
num_free_qubits
pop_condition
push_condition
reset
right_rotate
set_max_qubits
swap
unary_iteration
x
z
z_pow

KickMix module location:
/content/drive/MyDrive/Quantum_Competition/Extracted/kickmix/kickmix.so


In [ ]:
print("OperationStatistics methods:\n")

stats = km.Circuit.from_file(
    str(ROOT / "sample_submission/dqi.kmb"),
    format="kmb"
).operation_statistics()

print(stats)

print("\nAvailable statistics methods:")
print(dir(stats))

OperationStatistics methods:

<km.OperationStatistics num_instructions=86051014 num_toffoli_like=5725461 num_t_like=0>

Available statistics methods:
['__class__', '__contains__', '__delattr__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', 'get', 'items', 'keys', 'num_instructions', 'num_non_clifford', 'num_t_like', 'num_toffoli_like', 'num_z_pow', 'values', 'z_pow_clifford_hierarchy_counts']


In [ ]:
print("All public Circuit methods:\n")

for name in dir(km.Circuit):
    if not name.startswith("_"):
        print(name)

All public Circuit methods:

flame_chart_svg
from_file
html_diagram
num_bits
num_qubits
num_registers
operation_statistics
reaction_depth
register_data
text_diagram
to_file


In [ ]:
ROOT = Path("/content/drive/MyDrive/Quantum_Competition/Extracted")
p = ROOT / "sample_submission/dqi.kmb"

circuit = km.Circuit.from_file(str(p), format="kmb")

stats = circuit.operation_statistics()

print("RESOURCE BREAKDOWN")
print("------------------")

for key, value in stats.items():
    print(f"{key}: {value}")

print("\nReaction depth:", circuit.reaction_depth())
print("Qubits:", circuit.num_qubits)
print("Total operations:", len(circuit))

RESOURCE BREAKDOWN
------------------
NEG: 1302
BIT_INVERT: 6479824
BIT_STORE0: 689460
BIT_STORE1: 49140
X: 828388
Z: 33744
R: 1227624
HMR: 1227624
CX: 53871594
CZ: 9440886
SWAP: 3134247
CSWAP: 1172616
CCX: 4552845
INFO_TAG: 3257126
POP_CONDITION: 42297
PUSH_CONDITION: 42297

Reaction depth: 170757
Qubits: 1893
Total operations: 86051014


In [ ]:
for method in [
    km.CircuitBuilder.mark,
    km.CircuitBuilder.info_tag,
    km.CircuitBuilder.finish_circuit,
    km.Circuit.to_file
]:
    print(f"\n{method.__name__}")
    print(method.__doc__)


mark
mark(self: kickmix.CircuitBuilder, name: str) -> kickmix._CircuitBuilderMark

@signature def mark(self, name: str) -> Any:
Returns a context manager that emits @enter/@exit INFO_TAG instructions.

These INFO_TAG instructions are intended as debug information about entering
and exiting subroutines. For example, they are used by
`km.Circuit.flame_chart_svg()`.

Args:
    name: Label shown for the block in `builder.flame_chart_svg()`.

Returns:
    A context manager that marks blocks under `name`.

Examples:
    >>> import pathlib
    >>> import tempfile
    >>> import kickmix as km
    >>> builder = km.CircuitBuilder()
    >>> address = builder.create_quantum_register(4, name='address')
    >>> total = builder.create_quantum_register(8, name='total')
    >>> lookup_mark = builder.mark('iadd_lookup')
    >>> def iadd_lookup(table: list[int], idx: km.array, dst: km.array):
    ...     with lookup_mark:
    ...         buf = builder.alloc_qubits(len(dst))
    ...         builder.init_

In [ ]:
ROOT = Path("/content/drive/MyDrive/Quantum_Competition")
circuit = km.Circuit.from_file(
    str(ROOT / "Extracted/sample_submission/dqi.kmb"),
    format="kmb"
)

svg = circuit.flame_chart_svg()

print("Flame chart generated:", bool(svg))
print("SVG length:", len(svg))
print(svg[:500])

Flame chart generated: True
SVG length: 59267
<svg viewBox="-100 -140 1200.0 340.0" style="display: block; margin: auto;" xmlns="http://www.w3.org/2000/svg">
<rect x="0" y="-95.2985" width="0" height="95.2985" stroke="black" fill="blue" />
<rect x="0" y="0" width="0" height="10" stroke="black" fill="red" opacity="0.8" />
<text x="0" y="7" font-size="0.5" dominant-baseline="bottom" text-anchor="middle" fill="black" >gf_iadd (x4)</text>
<rect x="0" y="-45.2721" width="5.0616" height="45.2721" stroke="black" fill="blue" />
<rect x="0" y="20" w


In [ ]:
import xml.etree.ElementTree as ET
from collections import Counter

root_svg = ET.fromstring(svg)

labels = []

for element in root_svg.iter():
    if element.tag.endswith("text"):
        label = "".join(element.itertext()).strip()
        if label:
            labels.append(label)

print("SUBROUTINE LABELS")
print("=================")

for label in labels:
    print(label)

print("\nTotal labelled blocks:", len(labels))

SUBROUTINE LABELS
gf_iadd (x4)
left_rotate (x828)
c_iadd_classical (x69)
c_iadd_classical (x69)
c_isub_classical (x69)
flip_if_lt (x69)
shift_step (x69)
initial_shift
flip_if_lt (x444)
iadd_classical (x222)
iadd_classical (x222)
iadd_classical (x222)
iadd_classical (x222)
isub_classical (x222)
update_flag (x222)
gf_iadd (x2664)
linear_map (x2664)
gf_ifrobenius (x2664)
gf_idiv_x (x2220)
linear_map (x2220)
gf_idiv_classical (x2220)
gf_imul_x (x2220)
linear_map (x2220)
gf_imul_classical (x2220)
gf_mul (x2220)
linear_map (x1332)
gf_ifrobenius_adjoint (x1332)
gf_inverse_chain (x444)
gf_idiv_x (x444)
linear_map (x444)
gf_idiv_classical (x444)
gf_imul_x (x444)
linear_map (x444)
gf_imul_classical (x444)
gf_mul (x444)
gf_div (x444)
c_gf_iadd (x444)
gf_phase_by_product (x444)
gf_unmul (x444)
linear_map (x2664)
gf_ifrobenius_adjoint (x2664)
gf_phase_by_product (x2220)
gf_unmul (x2220)
gf_iadd (x2664)
linear_map (x1332)
gf_ifrobenius (x1332)
gf_uninverse_chain (x444)
gf_undiv (x444)
quotient_div (

In [ ]:
root_svg = ET.fromstring(svg)

rects = []
texts = []

for el in root_svg.iter():
    tag = el.tag.split("}")[-1]

    if tag == "rect":
        width = float(el.get("width", 0))
        x = float(el.get("x", 0))
        if width > 0:
            rects.append((width, x))

    elif tag == "text":
        x = float(el.get("x", 0))
        label = "".join(el.itertext()).strip()
        if label:
            texts.append((x, label))

print("LARGEST FLAME-CHART BLOCKS")

for width, x in sorted(rects, reverse=True)[:15]:
    nearest = min(texts, key=lambda item: abs(item[0] - x))
    print(f"{nearest[1]} | width={width:.2f} | x={x:.2f}")

LARGEST FLAME-CHART BLOCKS
gf_iadd (x4) | width=1000.00 | x=0.00
left_rotate (x12) | width=368.43 | x=626.17
left_rotate (x12) | width=368.43 | x=626.17
left_rotate (x12) | width=368.43 | x=626.17
left_rotate (x12) | width=368.43 | x=626.17
flip_if_lt (x69) | width=368.43 | x=5.40
flip_if_lt (x69) | width=368.43 | x=5.40
flip_if_lt (x69) | width=368.43 | x=5.40
flip_if_lt (x69) | width=368.43 | x=5.40
gf_iadd (x2664) | width=281.76 | x=706.28
gf_iadd (x2664) | width=281.76 | x=706.28
gf_iadd (x2664) | width=281.76 | x=11.14
gf_iadd (x2664) | width=281.76 | x=11.14
right_rotate (x12) | width=252.34 | x=373.83
right_rotate (x12) | width=252.34 | x=373.83


In [ ]:
p = ROOT / "verifiers/dqi/dqi_reference_circuit.py"
lines = p.read_text(errors="replace").splitlines()

for start, end in [(291, 377), (522, 610)]:
    print(f"\n--- LINES {start}-{end} ---")
    for i in range(start - 1, min(end, len(lines))):
        print(f"{i+1}: {lines[i]}")


--- LINES 291-377 ---
291:     def _multiply_and_add_or_sub(self, builder: km.CircuitBuilder,
292:                                  reg_a: Sequence[km.array], reg_b: Sequence[km.array],
293:                                  n_A: km.array, n_a: km.array, n_q: km.array,
294:                                  ctrl, it: int, *, is_adjoint: bool) -> None:
295:         """Computes the leading-coefficient ratio `q_i` and updates `reg_b ^= q_i * reg_a`."""
296:         c0_hc, _, _, c3_hc = ctrl
297:         gf = self.field
298:         m = len(reg_b)
299: 
300:         def accumulate_quotient(index: int, control) -> None:
301:             with builder.mark('quotient_div'):
302:                 ratio, scaffold = builder.init_gf2_div_with_scaffold(
303:                     reg_b[index], reg_a[index], field=gf)
304:                 builder.gf2_iadd(ratio, target=q_i, field=gf, control=control)
305:                 builder.del_gf2_div_with_scaffold(
306:                     reg_b[index], reg_a[ind

In [ ]:
print("GF2Field documentation:")
print(km.GF2Field.__doc__)

print("\nAvailable GF2Field methods:")
for name in dir(km.GF2Field):
    if not name.startswith("_"):
        print(name)

print("\nInverse synthesis documentation:")
print(km.CircuitBuilder.init_gf2_inverse_with_scaffold.__doc__)

GF2Field documentation:
A binary extension field GF(2^m) with polynomial basis arithmetic.

Elements are polynomials of degree less than m over GF(2), represented
in Python as non-negative integers where bit k is the coefficient of x^k.
Addition is bitwise XOR, and multiplication is polynomial multiplication
reduced modulo an irreducible polynomial of degree m.

Examples:
    >>> import kickmix as km
    >>> field = km.GF2Field(4)
    >>> field.degree
    4
    >>> hex(field.modulus)
    '0x13'
    >>> field.primitive_element
    2
    >>> field.mul(3, 5)
    15


Available GF2Field methods:
add
degree
div
frobenius
invert
is_element
is_irreducible
is_primitive_element
mod
modulus
mul
pow
primitive_element
square

Inverse synthesis documentation:
init_gf2_inverse_with_scaffold(self: kickmix.CircuitBuilder, input: object, *, target: object = 'alloc', field: object = None) -> object

@signature def init_gf2_inverse_with_scaffold(self, input: km.array, *, target: km.array | str = "alloc",

In [ ]:
import kickmix as km

for degree in [4, 8, 12]:
    field = km.GF2Field(degree)
    print(
        f"GF(2^{degree}) | "
        f"modulus={hex(field.modulus)} | "
        f"primitive={field.primitive_element}"
    )

GF(2^4) | modulus=0x13 | primitive=2
GF(2^8) | modulus=0x11d | primitive=2
GF(2^12) | modulus=0x1053 | primitive=2


In [ ]:
builder = km.CircuitBuilder()
field = km.GF2Field(12)

x = builder.alloc_qubits(12)
inv, scaffold = builder.init_gf2_inverse_with_scaffold(
    x, field=field
)

circuit = builder.finish_circuit()
stats = circuit.operation_statistics()

print("Inverse test")
print("Qubits:", circuit.num_qubits)
print("Operations:", stats.num_instructions)
print("Toffoli-like:", stats.num_toffoli_like)
print("Scaffold qubits:", len(scaffold))

Inverse test
Qubits: 72
Operations: 3460
Toffoli-like: 315
Scaffold qubits: 48


In [ ]:
field = km.GF2Field(12)

# Ordinary inverse
b1 = km.CircuitBuilder()
x1 = b1.alloc_qubits(12)
b1.init_gf2_inverse(x1, field=field)
c1 = b1.finish_circuit()

# Scaffold inverse
b2 = km.CircuitBuilder()
x2 = b2.alloc_qubits(12)
b2.init_gf2_inverse_with_scaffold(x2, field=field)
c2 = b2.finish_circuit()

for name, circuit in [
    ("Ordinary", c1),
    ("Scaffold", c2)
]:
    s = circuit.operation_statistics()
    print(name, "| Qubits:", circuit.num_qubits,
          "| Operations:", s.num_instructions,
          "| Toffoli-like:", s.num_toffoli_like)

Ordinary | Qubits: 84 | Operations: 5353 | Toffoli-like: 315
Scaffold | Qubits: 72 | Operations: 3460 | Toffoli-like: 315


In [ ]:
log = Path(
    "/content/drive/MyDrive/Quantum_Competition/Experiments/DQI/experiment_log.csv"
)

log.parent.mkdir(parents=True, exist_ok=True)

with open(log, "a", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "DQI_INV_001", "DQI", "inverse_scaffold",
        "BENCHMARKED", 72, 3460, 315,
        "", "", "GF2^12 scaffold vs ordinary; 35.4% fewer operations"
    ])

print("Saved:", log)

Saved: /content/drive/MyDrive/Quantum_Competition/Experiments/DQI/experiment_log.csv


In [ ]:
import subprocess

ROOT = Path("/content/drive/MyDrive/Quantum_Competition/Extracted")
VERIFIER = ROOT / "verifiers/dqi/verify_submission_dqi.py"
CIRCUIT = ROOT / "sample_submission/dqi.kmb"

print("Verifier exists:", VERIFIER.exists())
print("Baseline exists:", CIRCUIT.exists())

result = subprocess.run(
    ["python", str(VERIFIER), "--help"],
    capture_output=True, text=True
)

print(result.stdout[:2500])

Verifier exists: True
Baseline exists: True



In [ ]:
import sys

KICKMIX = ROOT / "kickmix"

if str(KICKMIX) not in sys.path:
    sys.path.insert(0, str(KICKMIX))


print("KickMix loaded:", km.__file__)

KickMix loaded: /content/drive/MyDrive/Quantum_Competition/Extracted/kickmix/kickmix.so


In [ ]:
env = {**os.environ, "PYTHONPATH": str(KICKMIX)}

p = subprocess.run(
    ["python", str(VERIFIER), "--help"],
    capture_output=True, text=True, env=env
)

print(p.stdout[:3000] or p.stderr[:3000])

usage: verify_submission_dqi.py [-h] [--format {auto,kmb,kmx}] [--m M] [--n N]
                                [--degree DEGREE] [--r R]
                                [--num-shots NUM_SHOTS]
                                [--batch-size BATCH_SIZE]
                                [--log2-delta LOG2_DELTA] [--salt SALT]
                                [--seed SEED] [--max-workers MAX_WORKERS]
                                circuit_file

Verifier for Benchmark II: DQI Reed-Solomon MBUC syndrome decoder (arXiv:2408.08292, arXiv:2510.10967).

For parameters `(m, n, b)` (`m` codeword positions, `n` syndromes correcting up to
`l = n // 2` errors over `GF(2^b)` using `km.GF2Field(b)` with primitive element
`alpha = km.GF2Field(b).primitive_element`):

Registers (little-endian, width `b`, pairwise disjoint, no other registers allowed):
    S0..S{n-1}  Quantum syndrome registers (`km.q`). Hold `S_i = sum_j e_j alpha^{(i+1) j}`.
    E0..E{m-1}  Classical MBUC outcome registers (`km.b`). Hold 

In [ ]:
env = {**os.environ, "PYTHONPATH": str(KICKMIX)}

cmd = [
    "python", str(VERIFIER),
    str(CIRCUIT),
    "--num-shots", "256",
    "--batch-size", "256",
    "--max-workers", "1"
]

p = subprocess.run(
    cmd, capture_output=True, text=True, env=env
)

print("Return code:", p.returncode)
print((p.stdout + "\n" + p.stderr)[-4000:])

Return code: 1
{
  "status": "FAIL",
  "error": "Effective error rate 0.2929 >= 1/16 at log2_delta=-128.0 (empirical=0.0000, value_errors=0/256, phase_errors=0/256)"
}




In [ ]:
inv_dir = ROOT / "verifiers/invsqrt"

print("Verifier files:")
for p in inv_dir.iterdir():
    print(p.name)

print("\nBaseline circuit:")
print((ROOT / "sample_submission/invsqrt.kmb").exists())

Verifier files:
__init__.py
nr_reference_circuit.py
verify_submission_invsqrt.py
__pycache__

Baseline circuit:
True


In [ ]:
INV_VERIFIER = inv_dir / "verify_submission_invsqrt.py"
INV_CIRCUIT = ROOT / "sample_submission/invsqrt.kmb"

p = subprocess.run(
    ["python", str(INV_VERIFIER), "--help"],
    capture_output=True, text=True,
    env={**os.environ, "PYTHONPATH": str(KICKMIX)}
)

print(p.stdout[:3000] or p.stderr[:3000])

usage: verify_submission_invsqrt.py [-h] [--format {auto,kmb,kmx}] [--n-p N_P]
                                    [--beta BETA]
                                    [--precision-bits PRECISION_BITS]
                                    [--shots-per-x SHOTS_PER_X] [--eta ETA]
                                    [--width WIDTH] [--max-x MAX_X]
                                    [--system-qubits SYSTEM_QUBITS]
                                    [--salt SALT] [--seed SEED]
                                    [--batch-size BATCH_SIZE]
                                    [--max-workers MAX_WORKERS]
                                    circuit_file

Verifier for Benchmark I: phasing by `beta / sqrt(x)` (arXiv:2308.12352).

A submission is a kickmix circuit implementing, for every `x` in `[1, max_x]`,

    |x>|0...0>  ->  exp(2 pi i beta / sqrt(x)) |x>|0...0>

The benchmark instance has `n_p = 10` bits per momentum component: `x = |nu - nu'|^2` with
`max_x = 3 (2^n_p - 2)^2 = 3,133,452`, `widt

In [ ]:
NR_FILE = inv_dir / "nr_reference_circuit.py"

print(NR_FILE.read_text()[:5000])

"""Reference Newton-Raphson circuit for Benchmark I (see `verify_submission_invsqrt.py`).

Implements the three-step algorithm of the paper with kickmix arithmetic, for `beta = 1`:

1. Normalize. Count the leading zero *pairs* `s` of `x` into a unary register and shift
   `x` in place to `x_norm = x 2^(2s)`, so that `M = x_norm / 2^(2 n_p)` lies in `[1, 4)`
   and `1 / sqrt(x) = 2^(s - n_p) / sqrt(M)`.
2. Initial guess. `y0 ~ 1 / sqrt(M)` from one of four cubic polynomials, chosen by the top
   three bits of `x_norm` (coefficients loaded by a Clifford network from two AND bits),
   evaluated by Horner's rule on `delta = M - {1, 1.5, 2, 3}` with column-truncated
   multipliers: `r1 = c2 - T(c3 delta)`, `r2 = c1 - T(r1 delta)`, `y0 = c0 - T(r2 delta)`.
3. Newton-Raphson. `p = T(y0 (3 - T(M T(y0^2)))) = 2 / sqrt(M)`, then de-normalize with
   a unary-controlled shift: `y1 = p 2^s`, so `y1 / 2^F = 1 / sqrt(x)` with
   `F = wp + n_p + 1` fractional bits of a turn.

`y1` is then used to phas

In [ ]:
text = NR_FILE.read_text()
print(text[5000:11000])

 np.array(prec.coefficient_table(), dtype=np.int64)
    c0, c1, c2, c3 = (table[np.where(top == 0, third, 2 + second), j] for j in range(4))
    delta = [(top & third, -1)] + [(bit(x_norm, width - 2 - m), -m) for m in range(2, min(d, width - 2) + 1)]
    y0, mult = c0, c3
    for acc in (c2, c1, c0):  # r1, r2, y0.
        y0 = mult = acc - product(delta, mult, wc, wc, wc)
        if ((y0 <= 0) | (y0 >= 2**wc)).any():
            raise OverflowError('Horner intermediate does not fit its register.')

    # 3. Newton-Raphson: y0^2 as diagonal bits plus doubled cross terms.
    y = {k: bit(y0, wc - k) for k in range(1, wc + 1)}  # Bit of weight 2^-k.
    square = np.zeros_like(xs)
    for k in range(1, wc + 1):
        if 2 * k <= wsq:
            square += y[k] << (wsq - 2 * k)
        for l in range(k + 1, wc + 1):
            if k + l - 1 <= wsq:
                square += (y[k] & y[l]) << (wsq - (k + l - 1))
    m_rows = [(bit(x_norm, i), i - width + 2) for i in range(width)]
    z = p

In [ ]:
import importlib
import importlib.util
import numpy as np
spec = importlib.util.spec_from_file_location("nr_reference", NR_FILE)
nr = importlib.util.module_from_spec(spec)
sys.modules["nr_reference"] = nr
spec.loader.exec_module(nr)

xs = np.unique(np.geomspace(1, 3133452, 20000).astype(np.int64))

for p in [
    nr.Precision(),
    nr.Precision(delta_bits=13),
    nr.Precision(coef_bits=16),
    nr.Precision(delta_bits=13, coef_bits=16)
]:
    err = nr.relative_error(10, p, xs)
    print(p, "| Max error:", f"{err.max():.3e}",
          "| Requirement:", f"{2**-24:.3e}")

Precision(delta_bits=14, coef_bits=17, square_bits=28, product_bits=28, result_bits=29) | Max error: 2.638e-08 | Requirement: 5.960e-08
Precision(delta_bits=13, coef_bits=17, square_bits=28, product_bits=28, result_bits=29) | Max error: 3.091e-08 | Requirement: 5.960e-08
Precision(delta_bits=14, coef_bits=16, square_bits=28, product_bits=28, result_bits=29) | Max error: 6.728e-08 | Requirement: 5.960e-08
Precision(delta_bits=13, coef_bits=16, square_bits=28, product_bits=28, result_bits=29) | Max error: 6.728e-08 | Requirement: 5.960e-08


In [ ]:
for p in [
    nr.Precision(delta_bits=13, square_bits=27),
    nr.Precision(delta_bits=13, product_bits=27),
    nr.Precision(delta_bits=13, result_bits=28),
    nr.Precision(delta_bits=13, square_bits=27, product_bits=27)
]:
    try:
        err = nr.relative_error(10, p, xs)
        print(p, "| Max error:", f"{err.max():.3e}",
              "| PASS:", err.max() < 2**-24)
    except Exception as e:
        print(p, "| Rejected:", str(e))

Precision(delta_bits=13, coef_bits=17, square_bits=27, product_bits=28, result_bits=29) | Max error: 4.730e-08 | PASS: True
Precision(delta_bits=13, coef_bits=17, square_bits=28, product_bits=27, result_bits=29) | Max error: 4.078e-08 | PASS: True
Precision(delta_bits=13, coef_bits=17, square_bits=28, product_bits=28, result_bits=28) | Max error: 3.867e-08 | PASS: True
Precision(delta_bits=13, coef_bits=17, square_bits=27, product_bits=27, result_bits=29) | Max error: 5.648e-08 | PASS: True


In [ ]:
text = NR_FILE.read_text()

for term in ["def build_nr_circuit", "def main", "add_argument"]:
    print(f"\n--- {term} ---")
    pos = text.find(term)
    print(text[pos:pos+1800] if pos >= 0 else "Not found")


--- def build_nr_circuit ---
def build_nr_circuit(n_p: int, *, prec: Precision = Precision(),
                     compute_only: bool = False) -> km.Circuit:
    """The reference circuit; `compute_only` stops after step 3 with `y1` named (for tests)."""
    if n_p < 1:
        raise ValueError(f'n_p must be positive, got {n_p}.')
    width = 2 * n_p + 2
    out_bits = prec.output_bits(n_p)
    builder = km.CircuitBuilder()

    def alloc(length: int, name: str) -> km.array:  # Named in compute_only mode, for tests.
        if compute_only:
            return builder.create_quantum_register(length, name=name)
        return builder.alloc_qubits(length)

    x = builder.create_quantum_register(width, name='x')
    y1, flags, selectors = alloc(out_bits, 'y1'), alloc(n_p, 'flags'), alloc(3, 'selectors')
    coefs = [alloc(prec.coef_bits, f'c{j}') for j in range(4)]
    square, z = alloc(prec.square_bits, 'square'), alloc(prec.product_bits + 2, 'z')
    scratch = alloc(max(prec.square_bits

In [ ]:
COMP_ROOT = Path("/content/drive/MyDrive/Quantum_Competition")
OUT = COMP_ROOT / "Experiments/INVSQRT/circuits"
OUT.mkdir(parents=True, exist_ok=True)

candidate_precision = nr.Precision(
    delta_bits=13,
    coef_bits=17,
    square_bits=28,
    product_bits=28,
    result_bits=28
)

candidate = nr.build_nr_circuit(10, prec=candidate_precision)

candidate_path = OUT / "invsqrt_precision_v1.kmb"
candidate.to_file(str(candidate_path), format="kmb")

stats = candidate.operation_statistics()

print("Candidate saved:", candidate_path)
print("Qubits:", candidate.num_qubits)
print("Operations:", stats.num_instructions)
print("Toffoli-like:", stats.num_toffoli_like)

Candidate saved: /content/drive/MyDrive/Quantum_Competition/Experiments/INVSQRT/circuits/invsqrt_precision_v1.kmb
Qubits: 259
Operations: 60407
Toffoli-like: 7533


In [ ]:
max_x = 3 * (2**10 - 2)**2
max_error = 0.0
worst_x = 0

for start in range(1, max_x + 1, 50000):
    end = min(start + 50000, max_x + 1)
    values = np.arange(start, end, dtype=np.int64)

    errors = nr.relative_error(10, candidate_precision, values)
    idx = int(np.argmax(errors))

    if errors[idx] > max_error:
        max_error = float(errors[idx])
        worst_x = int(values[idx])

print("Full-domain maximum error:", max_error)
print("Worst input:", worst_x)
print("Required maximum:", 2**-24)
print("PASS:", max_error < 2**-24)

Full-domain maximum error: 4.3657034409115057e-08
Worst input: 748950
Required maximum: 5.960464477539063e-08
PASS: True


In [ ]:
log = COMP_ROOT / "Experiments/INVSQRT/experiment_log.csv"

with open(log, "a", newline="") as f:
    csv.writer(f).writerow([
        "INVSQRT_V1",
        "INVSQRT",
        "reduced_precision",
        "MODEL_PASS",
        candidate.num_qubits,
        stats.num_instructions,
        stats.num_toffoli_like,
        "",
        max_error,
        "delta=13 coef=17 square=28 product=28 result=28; full-domain model passed"
    ])

print("V1 metrics saved:", log)

V1 metrics saved: /content/drive/MyDrive/Quantum_Competition/Experiments/INVSQRT/experiment_log.csv


In [ ]:
env = {**os.environ, "PYTHONPATH": str(KICKMIX)}

p = subprocess.run(
    ["python", str(INV_VERIFIER), "--help"],
    capture_output=True, text=True, env=env
)

print(p.stdout[-2500:])

--salt` when given). Shots are simulated across CPU cores in batches of
`batch_size`.

Cost (counted across simulated shots in `km.Simulator`):
    Toffolis     Mean Toffoli-like (`CCX`, `CCZ`, `CSWAP`) gates executed per shot.
    Z_POW        Clifford angles are free, T angles cost 1 T, other angles cost
                 `0.56 log2(1/eps) + 5.3` T each (mixed fallback synthesis,
                 arXiv:2203.10064), with `eps` the synthesis error in operator norm up
                 to global phase. That allows a relative phase error of up to `2 eps`
                 radians per rotation. Synthesis error isn't simulated, so the
                 worst case `n_rot * 2 eps` must fit in the tolerance left over by the
                 simulated error, `2 pi min_x (tol(x) - |error(x)|)` radians. The
                 verifier uses the largest `eps` that fits, split evenly.
    C            Toffolis + T / 4.
    Q            `circuit.num_qubits`.
    score        Q * C.

Usage:
    python invs

In [ ]:
env = {**os.environ, "PYTHONPATH": str(KICKMIX)}

cmd = [
    "python", str(INV_VERIFIER),
    str(candidate_path),
    "--max-x", "100",
    "--shots-per-x", "1",
    "--batch-size", "256",
    "--max-workers", "1"
]

p = subprocess.run(cmd, capture_output=True, text=True, env=env)

print("Return code:", p.returncode)
print((p.stdout + "\n" + p.stderr)[-3500:])

Return code: 1
{
  "status": "FAIL",
  "error": "num_shots must be a multiple of batch_size=256, got 100."
}




In [ ]:
cmd = [
    "python", str(INV_VERIFIER),
    str(candidate_path),
    "--max-x", "256",
    "--shots-per-x", "1",
    "--batch-size", "256",
    "--max-workers", "1"
]

p = subprocess.run(cmd, capture_output=True, text=True, env=env)

print("Return code:", p.returncode)
print((p.stdout + "\n" + p.stderr)[-2500:])

Return code: 0
{
  "status": "PASS",
  "score": 2005914.3063123147,
  "toffoli_equivalent": 7744.842881514729,
  "toffoli_count": 7533.0,
  "max_toffoli_count": 7533,
  "t_count": 1.0,
  "rotation_count": 36.0,
  "rotation_eps": 1.6254645325190662e-10,
  "num_qubits": 259,
  "reaction_depth": 3887,
  "num_ops": 60407,
  "verified_shots": 256,
  "worst_error_over_tolerance": 0.5,
  "worst_x": 1,
  "circuit_sha256": "de1d9c0cb7c059227f20b89f54172ee9bbd08818240320b6a1cbd296b64bb022"
}




In [ ]:
result = json.loads(p.stdout)
pilot_file = COMP_ROOT / "Experiments/INVSQRT/invsqrt_v1_pilot_result.json"
pilot_file.write_text(json.dumps(result, indent=2))

df_dir = ROOT / "verifiers/dfthcsos"

print("Pilot result saved:", pilot_file)
print("\nDFTHCSOS verifier files:")
for f in sorted(df_dir.rglob("*")):
    if f.is_file():
        print(f.relative_to(df_dir))

Pilot result saved: /content/drive/MyDrive/Quantum_Competition/Experiments/INVSQRT/invsqrt_v1_pilot_result.json

DFTHCSOS verifier files:
FeMoCo76o113e.fcidump
FeMoCo76o113e.fixed_shifts.npz
SCORING_README.md
__init__.py
__pycache__/__init__.cpython-313.pyc
example_submission/angle_k_reference_circuit.py
scoreinfra/__init__.py
scoreinfra/__pycache__/__init__.cpython-313.pyc
scoreinfra/__pycache__/circuits.cpython-313.pyc
scoreinfra/__pycache__/costs_prepare.cpython-313.pyc
scoreinfra/__pycache__/dfthc_math.cpython-313.pyc
scoreinfra/__pycache__/reference.cpython-313.pyc
scoreinfra/circuits.py
scoreinfra/costs_prepare.py
scoreinfra/dfthc_math.py
scoreinfra/reference.py
verify.py


In [ ]:
df_dir = ROOT / "verifiers/dfthcsos"

for filename in [
    "SCORING_README.md",
    "verify.py",
    "example_submission/angle_k_reference_circuit.py"
]:
    path = df_dir / filename
    print(f"\n{'='*25} {filename} {'='*25}")
    content = path.read_text()
    print(content[:12000])


========================= SCORING_README.md =========================
# `dfthcsos`: the DFTHC-SOS competition verifier

This package scores submissions to the Qaggle DFTHC-SOS competition. It is the
`verifier/` directory of the dfthc-sos repository, copied here and reduced to
the one system the competition is scored against:

| | |
| --- | --- |
| System | **FeMoCo76o113e**: `N = 76` spatial orbitals, `eta = 113` electrons |
| `E_upper` | `-22140.27335` Ha |
| Provided baseline | `R, B, C = 65, 60, 60` |
| Fixed symmetry shift | `S, beta_1` of the provided baseline, from [FeMoCo76o113e.fixed_shifts.npz](FeMoCo76o113e.fixed_shifts.npz) (SHA-256 `7b7facf63fe35a9d9d3d00787ffdb700c863fa845fe4724d7f7c54778151cd87`); submitted `H1_I` and `Hsym_lower` are ignored |
| Strict acceptance thresholds | Raw LCU normalization `Lambda > 49.0` and `E_gap > 3.914898` Ha; equality is rejected |
| Reference integrals | [FeMoCo76o113e.fcidump](FeMoCo76o113e.fcidump), shipped in this directory (128,128,34

In [ ]:
DF_BASE = ROOT / "sample_submission/dfthcsos"
DF_VERIFIER = ROOT / "verifiers/dfthcsos/verify.py"

env = {**os.environ, "PYTHONPATH": str(ROOT) + ":" + str(KICKMIX)}

p = subprocess.run(
    ["python", str(DF_VERIFIER), str(DF_BASE)],
    capture_output=True, text=True, env=env
)

print("Return code:", p.returncode)
print((p.stdout + "\n" + p.stderr)[-3000:])

Return code: 0
{
  "status": "PASS",
  "score": 3355599588.3155375,
  "lambda_eff": 49.30136925057439,
  "toffoli_count": 93237.0,
  "num_qubits": 730,
  "brot": 12,
  "batch_size": 8,
  "num_circuits": 10,
  "reaction_depth": 110,
  "num_ops": 2060743,
  "verified_shots": 254464,
  "submission_sha256": "276d464ff14da828b6b33e8487d99975494cbdf0c3cb2bd37dd3c3ddc4002695",
  "resources": {
    "brot": 12,
    "keep_bits": 10,
    "angles_toffoli_per_pass": 39740.0,
    "angles_toffoli": 79480.0,
    "angles_qubits": 119,
    "angles_selection_qubits": 12,
    "angles_persistent_qubits": 96,
    "angles_temporary_qubits": 11,
    "outer_state_prep_toffoli": 846,
    "inner_state_prep_toffoli": 9461,
    "givens_rotations_toffoli": 3450,
    "outer_state_prep_persistent_qubits": 46,
    "outer_state_prep_temporary_qubits": 370,
    "outer_state_prep_qubits": 416,
    "inner_state_prep_persistent_qubits": 44,
    "inner_state_prep_temporary_qubits": 544,
    "inner_state_prep_qubits": 588,
 

In [ ]:
from datetime import datetime

DF_EXP = COMP_ROOT / "Experiments/DFTHCSOS"
DF_CIRCUITS = DF_EXP / "circuits"
DF_CIRCUITS.mkdir(parents=True, exist_ok=True)
for src in DF_BASE.rglob("*"):
    if src.is_file():
        dst = DF_CIRCUITS / src.relative_to(DF_BASE)
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)

result = json.loads(p.stdout)

result["experiment"] = "DFTHCSOS_BASELINE"
result["status"] = "PASS"
result["timestamp"] = datetime.now().isoformat()

with open(DF_EXP / "baseline_result.json", "w") as f:
    json.dump(result, f, indent=2)

log_path = DF_EXP / "experiment_log.csv"
row = {
    "experiment": "DFTHCSOS_BASELINE",
    "status": "PASS",
    "score": result["score"],
    "lambda_eff": result["lambda_eff"],
    "toffoli_count": result["toffoli_count"],
    "num_qubits": result["num_qubits"],
    "num_ops": result["num_ops"],
    "reaction_depth": result["reaction_depth"],
    "note": "Official baseline verifier passed"
}

with open(log_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=row.keys())
    writer.writeheader()
    writer.writerow(row)

print("Baseline preserved:", DF_CIRCUITS)
print("Result and CSV log saved.")
print("Files copied:", sum(1 for x in DF_CIRCUITS.rglob("*") if x.is_file()))

Baseline preserved: /content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/circuits
Result and CSV log saved.
Files copied: 11


In [ ]:
BUILDER = ROOT / "verifiers/dfthcsos/example_submission/angle_k_reference_circuit.py"

p = subprocess.run(
    ["python", str(BUILDER), "--help"],
    capture_output=True, text=True,
    env=env
)

print(p.stdout)
print(p.stderr[-1000:])

usage: angle_k_reference_circuit.py [-h] --brot BROT [--batch-size BATCH_SIZE]
                                    [--format {kmx,kmb}] [--overwrite]
                                    angle_words output_directory

Write the ANGLE_k kickmix circuits for an angle-word table.

positional arguments:
  angle_words           submission.npz (an .npz with angle_words) or an .npy
  output_directory      receives angle_k_0.kmx, angle_k_1.kmx, ...

options:
  -h, --help            show this help message and exit
  --brot BROT           bits per word (the word register width)
  --batch-size BATCH_SIZE
                        words per circuit, J (default 8)
  --format {kmx,kmb}    circuit file format (default kmx)
  --overwrite           replace existing circuit files




In [ ]:
DF_CANDIDATE = DF_EXP / "candidate_J4"
DF_CANDIDATE.mkdir(parents=True, exist_ok=True)

ANGLE_WORDS = DF_BASE / "submission.npz"

p = subprocess.run(
    [
        "python", str(BUILDER),
        str(ANGLE_WORDS), str(DF_CANDIDATE),
        "--brot", "12",
        "--batch-size", "4",
        "--format", "kmb"
    ],
    capture_output=True, text=True, env=env
)

print("Return code:", p.returncode)
print((p.stdout + "\n" + p.stderr)[-2000:])

Return code: 1

Traceback (most recent call last):
  File "/content/drive/MyDrive/Quantum_Competition/Extracted/verifiers/dfthcsos/example_submission/angle_k_reference_circuit.py", line 230, in <module>
    main()
    ~~~~^^
  File "/content/drive/MyDrive/Quantum_Competition/Extracted/verifiers/dfthcsos/example_submission/angle_k_reference_circuit.py", line 224, in main
    paths = write_kmx_family(circuits, args.output_directory, format=args.format, overwrite=args.overwrite)
  File "/content/drive/MyDrive/Quantum_Competition/Extracted/verifiers/dfthcsos/example_submission/angle_k_reference_circuit.py", line 174, in write_kmx_family
    raise FileExistsError(f"{existing[0]} exists; pass overwrite=True (--overwrite) to replace it")
FileExistsError: /content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J4/angle_k_0.kmb exists; pass overwrite=True (--overwrite) to replace it



In [ ]:
DF_J4 = DF_EXP / "candidate_J4"

shutil.copy2(
    DF_BASE / "submission.npz",
    DF_J4 / "submission.npz"
)

p4 = subprocess.run(
    ["python", str(DF_VERIFIER), str(DF_J4)],
    capture_output=True, text=True, env=env
)

print("Return code:", p4.returncode)
print((p4.stdout + "\n" + p4.stderr)[-2500:])

Return code: 0
{
  "status": "PASS",
  "score": 5540116045.452662,
  "lambda_eff": 49.30136925057439,
  "toffoli_count": 164769.0,
  "num_qubits": 682,
  "brot": 12,
  "batch_size": 4,
  "num_circuits": 19,
  "reaction_depth": 209,
  "num_ops": 2207401,
  "verified_shots": 254464,
  "submission_sha256": "604118370de766f09a563ce7e856631606b60f9db925b02882eb390c41b21c00",
  "resources": {
    "brot": 12,
    "keep_bits": 10,
    "angles_toffoli_per_pass": 75506.0,
    "angles_toffoli": 151012.0,
    "angles_qubits": 71,
    "angles_selection_qubits": 12,
    "angles_persistent_qubits": 48,
    "angles_temporary_qubits": 11,
    "outer_state_prep_toffoli": 846,
    "inner_state_prep_toffoli": 9461,
    "givens_rotations_toffoli": 3450,
    "outer_state_prep_persistent_qubits": 46,
    "outer_state_prep_temporary_qubits": 370,
    "outer_state_prep_qubits": 416,
    "inner_state_prep_persistent_qubits": 44,
    "inner_state_prep_temporary_qubits": 544,
    "inner_state_prep_qubits": 588,
 

In [ ]:
DF_J16 = DF_EXP / "candidate_J16"
DF_J16.mkdir(parents=True, exist_ok=True)

p16 = subprocess.run(
    [
        "python", str(BUILDER),
        str(ANGLE_WORDS), str(DF_J16),
        "--brot", "12",
        "--batch-size", "16",
        "--format", "kmb"
    ],
    capture_output=True, text=True, env=env
)

print("Return code:", p16.returncode)
print((p16.stdout + "\n" + p16.stderr)[-1500:])

Return code: 1

Traceback (most recent call last):
  File "/content/drive/MyDrive/Quantum_Competition/Extracted/verifiers/dfthcsos/example_submission/angle_k_reference_circuit.py", line 230, in <module>
    main()
    ~~~~^^
  File "/content/drive/MyDrive/Quantum_Competition/Extracted/verifiers/dfthcsos/example_submission/angle_k_reference_circuit.py", line 224, in main
    paths = write_kmx_family(circuits, args.output_directory, format=args.format, overwrite=args.overwrite)
  File "/content/drive/MyDrive/Quantum_Competition/Extracted/verifiers/dfthcsos/example_submission/angle_k_reference_circuit.py", line 174, in write_kmx_family
    raise FileExistsError(f"{existing[0]} exists; pass overwrite=True (--overwrite) to replace it")
FileExistsError: /content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J16/angle_k_0.kmb exists; pass overwrite=True (--overwrite) to replace it



In [ ]:
shutil.copy2(
    DF_BASE / "submission.npz",
    DF_J16 / "submission.npz"
)

p16v = subprocess.run(
    ["python", str(DF_VERIFIER), str(DF_J16)],
    capture_output=True, text=True, env=env
)

print("Return code:", p16v.returncode)
print((p16v.stdout + "\n" + p16v.stderr)[-2500:])

Return code: 1
{
  "status": "FAIL",
  "error": "an ANGLE_k circuit uses 215 qubits > 128"
}




In [ ]:
DF_J10 = DF_EXP / "candidate_J10"
DF_J10.mkdir(parents=True, exist_ok=True)

p10 = subprocess.run(
    [
        "python", str(BUILDER),
        str(ANGLE_WORDS), str(DF_J10),
        "--brot", "12",
        "--batch-size", "10",
        "--format", "kmb"
    ],
    capture_output=True, text=True, env=env
)

print("Return code:", p10.returncode)
print((p10.stdout + "\n" + p10.stderr)[-1500:])

Return code: 1

Traceback (most recent call last):
  File "/content/drive/MyDrive/Quantum_Competition/Extracted/verifiers/dfthcsos/example_submission/angle_k_reference_circuit.py", line 230, in <module>
    main()
    ~~~~^^
  File "/content/drive/MyDrive/Quantum_Competition/Extracted/verifiers/dfthcsos/example_submission/angle_k_reference_circuit.py", line 224, in main
    paths = write_kmx_family(circuits, args.output_directory, format=args.format, overwrite=args.overwrite)
  File "/content/drive/MyDrive/Quantum_Competition/Extracted/verifiers/dfthcsos/example_submission/angle_k_reference_circuit.py", line 174, in write_kmx_family
    raise FileExistsError(f"{existing[0]} exists; pass overwrite=True (--overwrite) to replace it")
FileExistsError: /content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J10/angle_k_0.kmb exists; pass overwrite=True (--overwrite) to replace it



In [ ]:
DF_J9 = DF_EXP / "candidate_J9"
DF_J9.mkdir(parents=True, exist_ok=True)

p9 = subprocess.run(
    [
        "python", str(BUILDER),
        str(ANGLE_WORDS), str(DF_J9),
        "--brot", "12",
        "--batch-size", "9",
        "--format", "kmb"
    ],
    capture_output=True, text=True, env=env
)

print("Return code:", p9.returncode)
print((p9.stdout + "\n" + p9.stderr)[-1200:])

Return code: 1

Traceback (most recent call last):
  File "/content/drive/MyDrive/Quantum_Competition/Extracted/verifiers/dfthcsos/example_submission/angle_k_reference_circuit.py", line 230, in <module>
    main()
    ~~~~^^
  File "/content/drive/MyDrive/Quantum_Competition/Extracted/verifiers/dfthcsos/example_submission/angle_k_reference_circuit.py", line 224, in main
    paths = write_kmx_family(circuits, args.output_directory, format=args.format, overwrite=args.overwrite)
  File "/content/drive/MyDrive/Quantum_Competition/Extracted/verifiers/dfthcsos/example_submission/angle_k_reference_circuit.py", line 174, in write_kmx_family
    raise FileExistsError(f"{existing[0]} exists; pass overwrite=True (--overwrite) to replace it")
FileExistsError: /content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J9/angle_k_0.kmb exists; pass overwrite=True (--overwrite) to replace it



In [ ]:
DF_J7 = DF_EXP / "candidate_J7"

p7 = subprocess.run(
    ["python", str(BUILDER), str(ANGLE_WORDS), str(DF_J7),
     "--brot", "12", "--batch-size", "7",
     "--format", "kmb", "--overwrite"],
    capture_output=True, text=True, env=env
)

print("Return code:", p7.returncode)
print((p7.stdout + "\n" + p7.stderr)[-1000:])

Return code: 0
_J7/angle_k_2.kmb: 107 qubits, max_magic 3974
/content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J7/angle_k_3.kmb: 107 qubits, max_magic 3974
/content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J7/angle_k_4.kmb: 107 qubits, max_magic 3974
/content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J7/angle_k_5.kmb: 107 qubits, max_magic 3974
/content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J7/angle_k_6.kmb: 107 qubits, max_magic 3974
/content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J7/angle_k_7.kmb: 107 qubits, max_magic 3974
/content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J7/angle_k_8.kmb: 107 qubits, max_magic 3974
/content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J7/angle_k_9.kmb: 107 qubits, max_magic 3974
/content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J7/angle_k_10.kmb: 107 qubits, m

In [ ]:
p7 = subprocess.run(
    ["python", str(BUILDER), str(ANGLE_WORDS), str(DF_J7),
     "--brot", "12", "--batch-size", "7",
     "--format", "kmb", "--overwrite"],
    capture_output=True, text=True, env=env
)

print("Return code:", p7.returncode)
print((p7.stdout + "\n" + p7.stderr)[-1000:])

Return code: 0
_J7/angle_k_2.kmb: 107 qubits, max_magic 3974
/content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J7/angle_k_3.kmb: 107 qubits, max_magic 3974
/content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J7/angle_k_4.kmb: 107 qubits, max_magic 3974
/content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J7/angle_k_5.kmb: 107 qubits, max_magic 3974
/content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J7/angle_k_6.kmb: 107 qubits, max_magic 3974
/content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J7/angle_k_7.kmb: 107 qubits, max_magic 3974
/content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J7/angle_k_8.kmb: 107 qubits, max_magic 3974
/content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J7/angle_k_9.kmb: 107 qubits, max_magic 3974
/content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J7/angle_k_10.kmb: 107 qubits, m

In [ ]:
shutil.copy2(DF_BASE / "submission.npz", DF_J7 / "submission.npz")

p7v = subprocess.run(
    ["python", str(DF_VERIFIER), str(DF_J7)],
    capture_output=True, text=True, env=env
)

print("Return code:", p7v.returncode)
print((p7v.stdout + "\n" + p7v.stderr)[-2000:])

Return code: 0
{
  "status": "PASS",
  "score": 3581785396.1907077,
  "lambda_eff": 49.30136925057439,
  "toffoli_count": 101185.0,
  "num_qubits": 718,
  "brot": 12,
  "batch_size": 7,
  "num_circuits": 11,
  "reaction_depth": 121,
  "num_ops": 2018836,
  "verified_shots": 254464,
  "submission_sha256": "450edfcc49691a741c814b2cb9df08cb4d9b43a6e4280bbf6077e4dc1ba9d1d9",
  "resources": {
    "brot": 12,
    "keep_bits": 10,
    "angles_toffoli_per_pass": 43714.0,
    "angles_toffoli": 87428.0,
    "angles_qubits": 107,
    "angles_selection_qubits": 12,
    "angles_persistent_qubits": 84,
    "angles_temporary_qubits": 11,
    "outer_state_prep_toffoli": 846,
    "inner_state_prep_toffoli": 9461,
    "givens_rotations_toffoli": 3450,
    "outer_state_prep_persistent_qubits": 46,
    "outer_state_prep_temporary_qubits": 370,
    "outer_state_prep_qubits": 416,
    "inner_state_prep_persistent_qubits": 44,
    "inner_state_prep_temporary_qubits": 544,
    "inner_state_prep_qubits": 588,


In [ ]:
DF_J6 = DF_EXP / "candidate_J6"
DF_J6.mkdir(parents=True, exist_ok=True)

p6 = subprocess.run(
    ["python", str(BUILDER), str(ANGLE_WORDS), str(DF_J6),
     "--brot", "12", "--batch-size", "6", "--format", "kmb",
     "--overwrite"],
    capture_output=True, text=True, env=env
)

print("Return code:", p6.returncode)
print((p6.stdout + "\n" + p6.stderr)[-1000:])

Return code: 0
ndidate_J6/angle_k_4.kmb: 95 qubits, max_magic 3974
/content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J6/angle_k_5.kmb: 95 qubits, max_magic 3974
/content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J6/angle_k_6.kmb: 95 qubits, max_magic 3974
/content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J6/angle_k_7.kmb: 95 qubits, max_magic 3974
/content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J6/angle_k_8.kmb: 95 qubits, max_magic 3974
/content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J6/angle_k_9.kmb: 95 qubits, max_magic 3974
/content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J6/angle_k_10.kmb: 95 qubits, max_magic 3974
/content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J6/angle_k_11.kmb: 95 qubits, max_magic 3974
/content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J6/angle_k_12.kmb: 95 qubits, m

In [ ]:
shutil.copy2(DF_BASE / "submission.npz", DF_J6 / "submission.npz")

p6v = subprocess.run(
    ["python", str(DF_VERIFIER), str(DF_J6)],
    capture_output=True, text=True, env=env
)

print("Return code:", p6v.returncode)
print((p6v.stdout + "\n" + p6v.stderr)[-2000:])

Return code: 0
{
  "status": "PASS",
  "score": 4075211050.937909,
  "lambda_eff": 49.30136925057439,
  "toffoli_count": 117081.0,
  "num_qubits": 706,
  "brot": 12,
  "batch_size": 6,
  "num_circuits": 13,
  "reaction_depth": 143,
  "num_ops": 2097873,
  "verified_shots": 254464,
  "submission_sha256": "e6dbac99991595b05a685d342ad1bca84257053058b1b949513d18bdefe2443b",
  "resources": {
    "brot": 12,
    "keep_bits": 10,
    "angles_toffoli_per_pass": 51662.0,
    "angles_toffoli": 103324.0,
    "angles_qubits": 95,
    "angles_selection_qubits": 12,
    "angles_persistent_qubits": 72,
    "angles_temporary_qubits": 11,
    "outer_state_prep_toffoli": 846,
    "inner_state_prep_toffoli": 9461,
    "givens_rotations_toffoli": 3450,
    "outer_state_prep_persistent_qubits": 46,
    "outer_state_prep_temporary_qubits": 370,
    "outer_state_prep_qubits": 416,
    "inner_state_prep_persistent_qubits": 44,
    "inner_state_prep_temporary_qubits": 544,
    "inner_state_prep_qubits": 588,
 

In [ ]:
DF_J5 = DF_EXP / "candidate_J5"
DF_J5.mkdir(parents=True, exist_ok=True)

p5 = subprocess.run(
    ["python", str(BUILDER), str(ANGLE_WORDS), str(DF_J5),
     "--brot", "12", "--batch-size", "5", "--format", "kmb",
     "--overwrite"],
    capture_output=True, text=True, env=env
)

print("Return code:", p5.returncode)
print((p5.stdout + "\n" + p5.stderr)[-1000:])

Return code: 0
idate_J5/angle_k_6.kmb: 83 qubits, max_magic 3974
/content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J5/angle_k_7.kmb: 83 qubits, max_magic 3974
/content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J5/angle_k_8.kmb: 83 qubits, max_magic 3974
/content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J5/angle_k_9.kmb: 83 qubits, max_magic 3974
/content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J5/angle_k_10.kmb: 83 qubits, max_magic 3974
/content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J5/angle_k_11.kmb: 83 qubits, max_magic 3974
/content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J5/angle_k_12.kmb: 83 qubits, max_magic 3974
/content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J5/angle_k_13.kmb: 83 qubits, max_magic 3974
/content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/candidate_J5/angle_k_14.kmb: 83 qubits, m

In [ ]:
shutil.copy2(DF_BASE / "submission.npz", DF_J5 / "submission.npz")

p5v = subprocess.run(
    ["python", str(DF_VERIFIER), str(DF_J5)],
    capture_output=True, text=True, env=env
)

print("Return code:", p5v.returncode)
print((p5v.stdout + "\n" + p5v.stderr)[-2000:])

Return code: 0
{
  "status": "PASS",
  "score": 4549828036.110539,
  "lambda_eff": 49.30136925057439,
  "toffoli_count": 132977.0,
  "num_qubits": 694,
  "brot": 12,
  "batch_size": 5,
  "num_circuits": 15,
  "reaction_depth": 165,
  "num_ops": 2080128,
  "verified_shots": 254464,
  "submission_sha256": "2e7d8e31a43bc859fa6acf66dab9facb559c84114c35432366d1cbbcaedd2648",
  "resources": {
    "brot": 12,
    "keep_bits": 10,
    "angles_toffoli_per_pass": 59610.0,
    "angles_toffoli": 119220.0,
    "angles_qubits": 83,
    "angles_selection_qubits": 12,
    "angles_persistent_qubits": 60,
    "angles_temporary_qubits": 11,
    "outer_state_prep_toffoli": 846,
    "inner_state_prep_toffoli": 9461,
    "givens_rotations_toffoli": 3450,
    "outer_state_prep_persistent_qubits": 46,
    "outer_state_prep_temporary_qubits": 370,
    "outer_state_prep_qubits": 416,
    "inner_state_prep_persistent_qubits": 44,
    "inner_state_prep_temporary_qubits": 544,
    "inner_state_prep_qubits": 588,
 

In [ ]:
FINAL_DF = COMP_ROOT / "Final_Submission/dfthcsos"
FINAL_DF.mkdir(parents=True, exist_ok=True)

# Preserving J=8 verified baseline as the current winner
for src in DF_BASE.rglob("*"):
    if src.is_file():
        dst = FINAL_DF / src.relative_to(DF_BASE)
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

experiments = [
    ("J8_BASELINE", 8, 3355599588.3155375, 93237, 730, "PASS"),
    ("J7", 7, 3581785396.1907077, 101185, 718, "PASS"),
    ("J6", 6, 4075211050.937909, 117081, 706, "PASS"),
    ("J5", 5, 4549828036.110539, 132977, 694, "PASS"),
    ("J4", 4, 5540116045.452662, 164769, 682, "PASS"),
    ("J9", 9, None, None, 131, "FAIL_QUBIT_LIMIT"),
    ("J10", 10, None, None, 143, "FAIL_QUBIT_LIMIT"),
    ("J16", 16, None, None, 215, "FAIL_QUBIT_LIMIT")
]

log_path = DF_EXP / "experiment_log.csv"
with open(log_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["experiment", "batch_size", "score",
                     "toffoli_count", "qubits", "status"])
    writer.writerows(experiments)

print("Winner saved:", FINAL_DF)
print("Best score: 3,355,595,588.32")
print("Experiment log saved.")

Winner saved: /content/drive/MyDrive/Quantum_Competition/Final_Submission/dfthcsos
Best score: 3,355,595,588.32
Experiment log saved.


In [ ]:
p = subprocess.run(
    ["python", str(INV_VERIFIER), str(candidate_path),
     "--max-x", "4096",
     "--shots-per-x", "1",
     "--batch-size", "256",
     "--max-workers", "4"],
    capture_output=True, text=True,
    env={**os.environ, "PYTHONPATH": str(KICKMIX)}
)

print("Return code:", p.returncode)
print((p.stdout + "\n" + p.stderr)[-1500:])

Return code: 0
{
  "status": "PASS",
  "score": 2008688.8893046314,
  "toffoli_equivalent": 7755.555557160739,
  "toffoli_count": 7533.0,
  "max_toffoli_count": 7533,
  "t_count": 1.0,
  "rotation_count": 36.0,
  "rotation_eps": 3.72502288686517e-11,
  "num_qubits": 259,
  "reaction_depth": 3887,
  "num_ops": 60407,
  "verified_shots": 4096,
  "worst_error_over_tolerance": 0.6132812500107434,
  "worst_x": 729,
  "circuit_sha256": "de1d9c0cb7c059227f20b89f54172ee9bbd08818240320b6a1cbd296b64bb022"
}




In [ ]:
p = subprocess.run(
    ["python", str(INV_VERIFIER), str(candidate_path),
     "--min-x", "4097",
     "--max-x", "8192",
     "--shots-per-x", "1",
     "--batch-size", "256",
     "--max-workers", "4"],
    capture_output=True, text=True,
    env={**os.environ, "PYTHONPATH": str(KICKMIX)}
)

print("Return code:", p.returncode)
print((p.stdout + "\n" + p.stderr)[-1200:])

Return code: 2

usage: verify_submission_invsqrt.py [-h] [--format {auto,kmb,kmx}] [--n-p N_P]
                                    [--beta BETA]
                                    [--precision-bits PRECISION_BITS]
                                    [--shots-per-x SHOTS_PER_X] [--eta ETA]
                                    [--width WIDTH] [--max-x MAX_X]
                                    [--system-qubits SYSTEM_QUBITS]
                                    [--salt SALT] [--seed SEED]
                                    [--batch-size BATCH_SIZE]
                                    [--max-workers MAX_WORKERS]
                                    circuit_file
verify_submission_invsqrt.py: error: unrecognized arguments: --min-x 4097



In [ ]:
p = subprocess.run(
    ["python", str(INV_VERIFIER), str(candidate_path),
     "--max-x", "16384",
     "--shots-per-x", "1",
     "--batch-size", "256",
     "--max-workers", "4"],
    capture_output=True, text=True,
    env={**os.environ, "PYTHONPATH": str(KICKMIX)}
)

print("Return code:", p.returncode)
print((p.stdout + "\n" + p.stderr)[-1200:])

Return code: 0
{
  "status": "PASS",
  "score": 2009994.2493043924,
  "toffoli_equivalent": 7760.595557159816,
  "toffoli_count": 7533.0,
  "max_toffoli_count": 7533,
  "t_count": 1.0,
  "rotation_count": 36.0,
  "rotation_eps": 1.8625114436691207e-11,
  "num_qubits": 259,
  "reaction_depth": 3887,
  "num_ops": 60407,
  "verified_shots": 16384,
  "worst_error_over_tolerance": 0.6132812500107434,
  "worst_x": 729,
  "circuit_sha256": "de1d9c0cb7c059227f20b89f54172ee9bbd08818240320b6a1cbd296b64bb022"
}




In [ ]:
p = subprocess.run(
    ["python", str(INV_VERIFIER), str(candidate_path),
     "--max-x", "65536",
     "--shots-per-x", "1",
     "--batch-size", "256",
     "--max-workers", "4"],
    capture_output=True, text=True,
    env={**os.environ, "PYTHONPATH": str(KICKMIX)}
)

print("Return code:", p.returncode)
print((p.stdout + "\n" + p.stderr)[-1000:])

Return code: 0
{
  "status": "PASS",
  "score": 2011544.3128069323,
  "toffoli_equivalent": 7766.580358327924,
  "toffoli_count": 7533.0,
  "max_toffoli_count": 7533,
  "t_count": 1.0,
  "rotation_count": 36.0,
  "rotation_eps": 8.177822987738336e-12,
  "num_qubits": 259,
  "reaction_depth": 3887,
  "num_ops": 60407,
  "verified_shots": 65536,
  "worst_error_over_tolerance": 0.6589538755214376,
  "worst_x": 47055,
  "circuit_sha256": "de1d9c0cb7c059227f20b89f54172ee9bbd08818240320b6a1cbd296b64bb022"
}




In [ ]:
p = subprocess.run(
    ["python", str(INV_VERIFIER), str(candidate_path),
     "--max-x", "262144",
     "--shots-per-x", "1",
     "--batch-size", "256",
     "--max-workers", "4"],
    capture_output=True, text=True,
    env={**os.environ, "PYTHONPATH": str(KICKMIX)}
)

print("Return code:", p.returncode)
print((p.stdout + "\n" + p.stderr)[-1000:])

Return code: 0
{
  "status": "PASS",
  "score": 2013030.1448230916,
  "toffoli_equivalent": 7772.317161479118,
  "toffoli_count": 7533.0,
  "max_toffoli_count": 7533,
  "t_count": 1.0,
  "rotation_count": 36.0,
  "rotation_eps": 3.715257593536826e-12,
  "num_qubits": 259,
  "reaction_depth": 3887,
  "num_ops": 60407,
  "verified_shots": 262144,
  "worst_error_over_tolerance": 0.6912517396245655,
  "worst_x": 186847,
  "circuit_sha256": "de1d9c0cb7c059227f20b89f54172ee9bbd08818240320b6a1cbd296b64bb022"
}




In [ ]:
p = subprocess.run(
    ["python", str(INV_VERIFIER), str(candidate_path),
     "--max-x", "1048576",
     "--shots-per-x", "1",
     "--batch-size", "256",
     "--max-workers", "4"],
    capture_output=True, text=True,
    env={**os.environ, "PYTHONPATH": str(KICKMIX)}
)

print("Return code:", p.returncode)
print((p.stdout + "\n" + p.stderr)[-1000:])

Return code: 0
{
  "status": "PASS",
  "score": 2014607.1416426576,
  "toffoli_equivalent": 7778.4059522882535,
  "toffoli_count": 7533.0,
  "max_toffoli_count": 7533,
  "t_count": 1.0,
  "rotation_count": 36.0,
  "rotation_eps": 1.608113045825999e-12,
  "num_qubits": 259,
  "reaction_depth": 3887,
  "num_ops": 60407,
  "verified_shots": 1048576,
  "worst_error_over_tolerance": 0.7324434964698643,
  "worst_x": 748950,
  "circuit_sha256": "de1d9c0cb7c059227f20b89f54172ee9bbd08818240320b6a1cbd296b64bb022"
}




In [ ]:
p = subprocess.run(
    ["python", str(INV_VERIFIER), str(candidate_path),
     "--max-x", "3133452",
     "--shots-per-x", "1",
     "--batch-size", "4",
     "--max-workers", "4"],
    capture_output=True, text=True,
    env={**os.environ, "PYTHONPATH": str(KICKMIX)}
)

print("Return code:", p.returncode)
print((p.stdout + "\n" + p.stderr)[-1200:])

Return code: 0
{
  "status": "PASS",
  "score": 2015912.5016481976,
  "toffoli_equivalent": 7783.445952309644,
  "toffoli_count": 7533.0,
  "max_toffoli_count": 7533,
  "t_count": 1.0,
  "rotation_count": 36.0,
  "rotation_eps": 8.040565205476374e-13,
  "num_qubits": 259,
  "reaction_depth": 3887,
  "num_ops": 60407,
  "verified_shots": 3133452,
  "worst_error_over_tolerance": 0.7324434970463661,
  "worst_x": 2995800,
  "circuit_sha256": "de1d9c0cb7c059227f20b89f54172ee9bbd08818240320b6a1cbd296b64bb022"
}




In [ ]:
final_dir = COMP_ROOT / "Final_Submission"
final_dir.mkdir(parents=True, exist_ok=True)

final_inv = final_dir / "invsqrt.kmb"
shutil.copy2(candidate_path, final_inv)

result = {
    "status": "PASS",
    "verification": "full-domain, 1 shot per input",
    "verified_shots": 3133452,
    "score": 2015912.5016481976,
    "num_qubits": 259,
    "toffoli_count": 7533,
    "num_ops": 60407,
    "worst_error_over_tolerance": 0.7324434970463661,
    "worst_x": 2995800,
    "circuit_sha256": "de1d9c0cb7c059227f20b89f54172ee9bbd08818240320b6a1cbd296b64bb022"
}

with open(COMP_ROOT / "Experiments/INVSQRT/full_domain_screening.json", "w") as f:
    json.dump(result, f, indent=2)

print("Saved:", final_inv)
print("Size (MB):", round(final_inv.stat().st_size / 1e6, 2))

Saved: /content/drive/MyDrive/Quantum_Competition/Final_Submission/invsqrt.kmb
Size (MB): 0.56


In [ ]:
dqi_ref = ROOT / "verifiers/dqi/dqi_reference_circuit.py"

tree = ast.parse(dqi_ref.read_text())

for node in ast.walk(tree):
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        if any(word in node.name.lower() for word in
               ["inverse", "decoder", "forney", "chien", "eea", "build"]):
            print(node.name, "line:", node.lineno)

build_reference_circuit line: 618
build_syndrome_decoding_mbuc line: 73
_synchronized_eea line: 166
_chien_search_and_forney_mbuc line: 522


In [ ]:
lines = dqi_ref.read_text().splitlines()

for start, end in [(73,165), (166,260), (522,617), (618,700)]:
    print(f"\n{'='*20} LINES {start}-{end} {'='*20}")
    for i in range(start-1, min(end, len(lines))):
        print(f"{i+1}: {lines[i]}")


==================== LINES 73-165 ====================
73:     def build_syndrome_decoding_mbuc(self, builder: km.CircuitBuilder, *,
74:                                      syndromes: Sequence[km.array],
75:                                      error_mbuc: Sequence[km.array],
76:                                      n_iters: int) -> None:
77:         """Appends the syndrome-decoding MBUC phase circuit to `builder`.
78: 
79:         Args:
80:             builder: Circuit builder receiving the instructions.
81:             syndromes: `n_syndromes` quantum registers of width `field.degree`.
82:                 Restored to their initial state on completion.
83:             error_mbuc: Classical bit registers of width `field.degree`, one per
84:                 codeword position, holding X-basis measurement outcomes.
85:             n_iters: Number of synchronized EEA iterations (`default_n_iters(n)`).
86:         """
87:         gf = self.field
88:         n = self.n_syndromes
89:       

In [ ]:
for node in ast.walk(tree):
    if isinstance(node, ast.FunctionDef):
        print(f"{node.name}: line {node.lineno}")

_active: line 36
_xor_threshold: line 50
default_n_iters: line 613
build_reference_circuit: line 618
main: line 628
__init__: line 67
build_syndrome_decoding_mbuc: line 73
_initial_shift_loop: line 140
_synchronized_eea: line 166
_counter_controls: line 218
_del_counter_controls: line 230
_update_counter: line 240
_update_flag: line 250
_multiply_and_add_or_sub: line 291
_shift_b_array_right: line 378
_update_metadata: line 440
_swap_a_and_b: line 468
_update_halting_counter: line 483
_pre_align_reg_b: line 509
_chien_search_and_forney_mbuc: line 522
accumulate_quotient: line 300
sweep_positions: line 309
endpoint: line 405
endpoint: line 325


In [ ]:
lines = dqi_ref.read_text().splitlines()

for start, end in [(291,377), (378,439)]:
    print(f"\n--- {start}-{end} ---")
    for i in range(start-1, end):
        print(f"{i+1}: {lines[i]}")


--- 291-377 ---
291:     def _multiply_and_add_or_sub(self, builder: km.CircuitBuilder,
292:                                  reg_a: Sequence[km.array], reg_b: Sequence[km.array],
293:                                  n_A: km.array, n_a: km.array, n_q: km.array,
294:                                  ctrl, it: int, *, is_adjoint: bool) -> None:
295:         """Computes the leading-coefficient ratio `q_i` and updates `reg_b ^= q_i * reg_a`."""
296:         c0_hc, _, _, c3_hc = ctrl
297:         gf = self.field
298:         m = len(reg_b)
299: 
300:         def accumulate_quotient(index: int, control) -> None:
301:             with builder.mark('quotient_div'):
302:                 ratio, scaffold = builder.init_gf2_div_with_scaffold(
303:                     reg_b[index], reg_a[index], field=gf)
304:                 builder.gf2_iadd(ratio, target=q_i, field=gf, control=control)
305:                 builder.del_gf2_div_with_scaffold(
306:                     reg_b[index], reg_a[index], t

In [ ]:
for i in range(35, 73):
    print(f"{i+1}: {lines[i]}")

36: def _active(cursor: km.UnaryIterationCursor, address_value: int, limit: int):
37:     """Moves `cursor` to `address_value` and returns `control and (x == address_value)`.
38: 
39:     When `address_value > limit` (outside the caller's upper bound `x <= limit`),
40:     parks `cursor` out of range and returns `False` so controlled operations
41:     become no-ops.
42:     """
43:     if address_value > limit:
44:         cursor.move_to(_UNREACHABLE)
45:         return False
46:     cursor.move_to(address_value)
47:     return cursor.match_qubit
48: 
49: 
50: def _xor_threshold(builder: km.CircuitBuilder, cursor: km.UnaryIterationCursor,
51:                    limit: int, target, control, value: int, *,
52:                    at_least: bool, lowest: int = 0) -> None:
53:     """Initializes or clears `target ^= control and (x >= value)` (or `x < value`)."""
54:     if value > limit:
55:         if not at_least:
56:             builder.cx(control, target)
57:         return
58:     if 

In [ ]:
for name in dir(km.CircuitBuilder):
    if any(x in name.lower() for x in
           ["less", "greater", "compare", "threshold", "unary", "equal"]):
        print(name)

flip_if_equal
flip_if_greater_than
flip_if_less_than
unary_iteration


In [ ]:
s = test_circuit.operation_statistics()

print("Qubits:", test_circuit.num_qubits)
print(s)
print("Available statistics:", dir(s))

Qubits: 8
<km.OperationStatistics num_instructions=40 num_toffoli_like=3 num_t_like=0>
Available statistics: ['__class__', '__contains__', '__delattr__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', 'get', 'items', 'keys', 'num_instructions', 'num_non_clifford', 'num_t_like', 'num_toffoli_like', 'num_z_pow', 'values', 'z_pow_clifford_hierarchy_counts']


In [ ]:
print("LESS THAN:\n", km.CircuitBuilder.flip_if_less_than.__doc__)
print("GREATER THAN:\n", km.CircuitBuilder.flip_if_greater_than.__doc__)
print("EQUAL:\n", km.CircuitBuilder.flip_if_equal.__doc__)

LESS THAN:
 flip_if_less_than(self: kickmix.CircuitBuilder, lhs: object, rhs: object, *, target: object, or_equal: object = False, control: object = True, btol: float = inf) -> None

@signature def flip_if_less_than(self, lhs: km.array, rhs: km.array | int, *, target: km.q, or_equal: km.q | km.b | bool = False, control: km.q | km.b | bool = True, btol: float = float('inf')) -> None:
Appends operations to perform `if control: target ^= lhs < rhs + or_equal`.

Xors the result of an unsigned integer comparison into a target qubit.

Args:
    lhs: The little-endian value on the left hand side of the comparison.
    rhs: The little-endian value or non-negative integer constant on the
        right hand side of the comparison.
    target: The qubit to flip when lhs < rhs.
    or_equal: Defaults to False. Determines if the comparison is a
        less-than-or-equal-to test rather than a less-than test.
    control: Defaults to True. Determines if the comparisons happens at all.
    btol: Defa

In [ ]:
# Comparing native flip_if_less_than against the reference threshold logic

for value in range(8):
    b = km.CircuitBuilder()
    x = b.create_quantum_register(3, name="x")
    t = b.alloc_qubits(1)[0]

    b.flip_if_less_than(x, value, target=t)
    c = b.finish_circuit()

    print(
        f"value={value}: "
        f"instructions={c.operation_statistics().num_instructions}, "
        f"toffoli={c.operation_statistics().num_toffoli_like}"
    )

value=0: instructions=19, toffoli=2
value=1: instructions=25, toffoli=2
value=2: instructions=26, toffoli=2
value=3: instructions=32, toffoli=2
value=4: instructions=26, toffoli=2
value=5: instructions=32, toffoli=2
value=6: instructions=33, toffoli=2
value=7: instructions=39, toffoli=2


In [ ]:
print("\n".join(lines[45:68]))
print("\n".join(lines[210:240]))

    cursor.move_to(address_value)
    return cursor.match_qubit


def _xor_threshold(builder: km.CircuitBuilder, cursor: km.UnaryIterationCursor,
                   limit: int, target, control, value: int, *,
                   at_least: bool, lowest: int = 0) -> None:
    """Initializes or clears `target ^= control and (x >= value)` (or `x < value`)."""
    if value > limit:
        if not at_least:
            builder.cx(control, target)
        return
    if at_least:
        builder.cx(control, target)
    for u in range(lowest, value):
        builder.cx(_active(cursor, u, limit), target)


class ProosAndZalkaDecoder:
    """Builds the reversible Reed-Solomon MBUC decoder circuit (arXiv:2510.10967)."""

    def __init__(self, n_syndromes: int, field: km.GF2Field):
        if n_syndromes < 2:
                                                 flag, is_c3, is_adjoint=False)
                builder.del_and(counter[0], counter[1], is_c3)
                builder.free(is_c3)
             

In [ ]:
import inspect

print(km.UnaryIterationCursor.__doc__)
print(dir(km.UnaryIterationCursor))

A cursor that steps through values of an address register.

Create a cursor with `builder.unary_iteration(address, ...)` and open a pass
using a `with` block. Inside the block, iterate with a `for` loop to visit
`(address_value, match_qubit)` pairs in order, or call `move_to` to jump to
any value.

Entering the `with` block allocates `max(1, len(address))` temporary qubits;
leaving the block uncomputes and frees them. A cursor that is never opened
adds no operations to the circuit, and the same cursor can be reused across
multiple `with` blocks.

Examples:
    >>> import kickmix as km
    >>> builder = km.CircuitBuilder()
    >>> address = builder.create_quantum_register(2, name='address')
    >>> target = builder.create_quantum_register(1, name='target')
    >>> cursor = builder.unary_iteration(address)
    >>> with cursor:
    ...     for address_value, match_qubit in cursor:
    ...         if address_value == 3:
    ...             builder.cx(match_qubit, target[0])
    >>> # Reusi

In [ ]:
spec = importlib.util.spec_from_file_location("dqi_reference", dqi_ref)
dqi = importlib.util.module_from_spec(spec)
sys.modules["dqi_reference"] = dqi
spec.loader.exec_module(dqi)

print(inspect.getsource(dqi._active))
print(inspect.getsource(dqi.ProosAndZalkaDecoder._counter_controls))

def _active(cursor: km.UnaryIterationCursor, address_value: int, limit: int):
    """Moves `cursor` to `address_value` and returns `control and (x == address_value)`.

    When `address_value > limit` (outside the caller's upper bound `x <= limit`),
    parks `cursor` out of range and returns `False` so controlled operations
    become no-ops.
    """
    if address_value > limit:
        cursor.move_to(_UNREACHABLE)
        return False
    cursor.move_to(address_value)
    return cursor.match_qubit

    @staticmethod
    def _counter_controls(builder: km.CircuitBuilder, counter: km.array, is_c3,
                          is_hc_zero) -> km.array:
        """One-hot decodes `counter == 0..3` gated by `is_hc_zero` using 3 Toffolis."""
        ctrl = builder.alloc_qubits(4)
        c0, c1, c2, c3 = ctrl
        builder.init_and(is_hc_zero, km.array([counter[0], is_c3]), km.array([c1, c3]))
        builder.cx(km.array([is_hc_zero, c1]), c0)
        builder.init_and(c0, counter[1], c2)
   

In [ ]:
for node in ast.walk(tree):
    if isinstance(node, ast.Call) and getattr(node.func, "id", "") == "_xor_threshold":
        print(ast.get_source_segment("\n".join(lines), node))
        print("---")

_xor_threshold(builder, cur_B_ge, lim_B, do_shift, ctrl_nB, m - i + 1,
                                   at_least=True)
---
_xor_threshold(builder, cur_b_ge, lim_b, do_shift, c2_hc, i,
                                   at_least=True)
---
_xor_threshold(builder, cur_B_lt, lim_B, do_shift, c3_nf, m - i,
                                   at_least=False)
---
_xor_threshold(builder, cur_a, lim_a, mul, c3_hc, i + 1,
                                       at_least=True, lowest=1)
---
_xor_threshold(builder, cur_A, lim_A, mul, c0_hc, m - i,
                                       at_least=True, lowest=1)
---


In [ ]:
for i, line in enumerate(lines):
    if "cur_B_ge =" in line or "cur_b_ge =" in line or "cur_B_lt =" in line or "cur_a =" in line or "cur_A =" in line:
        print(f"{i+1}: {line.strip()}")

In [ ]:
for i, line in enumerate(lines):
    if any(name in line and "=" in line for name in
           ["cur_B_ge", "cur_b_ge", "cur_B_lt", "cur_a", "cur_A"]):
        print(f"{i+1}: {line.strip()}")

320: builder.unary_iteration(n_a, control=c3_hc) as cur_a,
321: builder.unary_iteration(n_A, control=c0_hc) as cur_A,
399: builder.unary_iteration(n_B, control=ctrl_nB) as cur_B_ge,
400: builder.unary_iteration(n_b, control=c2_hc) as cur_b_ge,
401: builder.unary_iteration(n_B, control=c3_nf) as cur_B_lt,


In [ ]:
print("\n".join(lines[310:330]))
print("\n".join(lines[390:410]))

                lim_q = min(m - 1, 1 + it)
                lim_q1 = min(m, 1 + it)
                lim_a = min(m - 2, it // 2 + 1)
                lim_A = m - 2
                order = range(m) if is_adjoint else range(m - 1, -1, -1)

                with (
                    builder.unary_iteration(n_q, control=c0_hc) as eq_i,
                    builder.unary_iteration(n_q, control=c3_hc) as eq_ip1,
                    builder.unary_iteration(n_a, control=c3_hc) as cur_a,
                    builder.unary_iteration(n_A, control=c0_hc) as cur_A,
                ):
                    mul = builder.alloc_qubits(1)[0]

                    def endpoint(i: int) -> None:
                        _xor_threshold(builder, cur_a, lim_a, mul, c3_hc, i + 1,
                                       at_least=True, lowest=1)
                        _xor_threshold(builder, cur_A, lim_A, mul, c0_hc, m - i,
                                       at_least=True, lowest=1)

            builder.x(flag)
    

In [ ]:
def native_threshold(builder, address, target, control,
                     value, limit, at_least, lowest=0):

    if value > limit:
        if not at_least:
            builder.cx(control, target)
        return

    if at_least:
        builder.flip_if_greater_than(
            address, value - 1,
            target=target, control=control
        )
        if lowest == 1:
            builder.flip_if_equal(
                address, 0,
                target=target, control=control
            )
    else:
        builder.flip_if_less_than(
            address, value,
            target=target, control=control
        )

print("Native threshold helper ready.")

Native threshold helper ready.


In [ ]:
def test_threshold(value, at_least, lowest=0):
    b = km.CircuitBuilder()
    address = b.create_quantum_register(3, name="address")
    target = b.create_quantum_register(1, name="target")

    native_threshold(
        b, address, target[0], True,
        value, 7, at_least, lowest
    )
    circuit = b.finish_circuit()

    sim = km.Simulator(batch_size=8)
    sim.use_same_registers_as(circuit)

    for xval in range(8):
        sim.write_within_shot("address", xval, xval)

    sim.do(circuit)

    for xval in range(8):
        actual = sim.read_within_shot("target", xval, out=int)
        expected = (
            int(xval == 0 or xval >= value) if at_least and lowest == 1
            else int(xval >= value) if at_least
            else int(xval < value)
        )
        if actual != expected:
            return False, xval, actual, expected

    return True, None, None, None

for low in (0, 1):
    for value in range(1, 8):
        print("at_least", low, value, test_threshold(value, True, low))

for value in range(8):
    print("less_than", value, test_threshold(value, False))

at_least 0 1 (True, None, None, None)
at_least 0 2 (True, None, None, None)
at_least 0 3 (True, None, None, None)
at_least 0 4 (True, None, None, None)
at_least 0 5 (True, None, None, None)
at_least 0 6 (True, None, None, None)
at_least 0 7 (True, None, None, None)
at_least 1 1 (True, None, None, None)
at_least 1 2 (True, None, None, None)
at_least 1 3 (True, None, None, None)
at_least 1 4 (True, None, None, None)
at_least 1 5 (True, None, None, None)
at_least 1 6 (True, None, None, None)
at_least 1 7 (True, None, None, None)
less_than 0 (True, None, None, None)
less_than 1 (True, None, None, None)
less_than 2 (True, None, None, None)
less_than 3 (True, None, None, None)
less_than 4 (True, None, None, None)
less_than 5 (True, None, None, None)
less_than 6 (True, None, None, None)
less_than 7 (True, None, None, None)


In [ ]:
def compare_threshold_cost(width, value, at_least, lowest=0):
    results = {}

    # Original unary implementation
    b = km.CircuitBuilder()
    address = b.create_quantum_register(width, name="address")
    target = b.create_quantum_register(1, name="target")
    cursor = b.unary_iteration(address)

    with cursor:
        if value > (2**width - 1):
            if not at_least:
                b.cx(True, target[0])
        else:
            if at_least:
                b.cx(True, target[0])
            for u in range(lowest, value):
                cursor.move_to(u)
                b.cx(cursor.match_qubit, target[0])

    old = b.finish_circuit().operation_statistics()

    # Native implementation
    b = km.CircuitBuilder()
    address = b.create_quantum_register(width, name="address")
    target = b.create_quantum_register(1, name="target")

    native_threshold(
        b, address, target[0], True,
        value, 2**width - 1, at_least, lowest
    )
    new = b.finish_circuit().operation_statistics()

    return (
        (old.num_instructions, old.num_toffoli_like),
        (new.num_instructions, new.num_toffoli_like)
    )

for value in (100, 500, 1000, 2000, 3000):
    print(value, compare_threshold_cost(12, value, True))

100 ((1060, 106), (121, 11))
500 ((5036, 503), (142, 11))
1000 ((10028, 1002), (149, 11))
2000 ((20020, 2001), (156, 11))
3000 ((30020, 3001), (156, 11))


In [ ]:
import ast, inspect, importlib.util, sys

source = dqi_ref.read_text()
src_lines = source.splitlines()

# Replace only the helper function
tree_candidate = ast.parse(source)
node = next(
    n for n in tree_candidate.body
    if isinstance(n, ast.FunctionDef) and n.name == "_xor_threshold"
)

replacement = '''
def _xor_threshold(builder, address, cursor, limit, target, control,
                   value, *, at_least, lowest=0):
    if value > limit:
        if not at_least:
            builder.cx(control, target)
        return

    if at_least:
        builder.flip_if_greater_than(
            address, value - 1, target=target, control=control
        )
        if lowest == 1:
            builder.flip_if_equal(
                address, 0, target=target, control=control
            )
    else:
        builder.flip_if_less_than(
            address, value, target=target, control=control
        )
'''

src_lines[node.lineno - 1:node.end_lineno] = replacement.strip("\\n").splitlines()
patched = "\n".join(src_lines) + "\n"

# Pass the correct address register at each call site
for old, new in [
    ("_xor_threshold(builder, cur_a,", "_xor_threshold(builder, n_a, cur_a,"),
    ("_xor_threshold(builder, cur_A,", "_xor_threshold(builder, n_A, cur_A,"),
    ("_xor_threshold(builder, cur_B_ge,", "_xor_threshold(builder, n_B, cur_B_ge,"),
    ("_xor_threshold(builder, cur_b_ge,", "_xor_threshold(builder, n_b, cur_b_ge,"),
    ("_xor_threshold(builder, cur_B_lt,", "_xor_threshold(builder, n_B, cur_B_lt,"),
]:
    patched = patched.replace(old, new)

candidate_source = DF_EXP / "dqi_reference_native_candidate.py"
candidate_source.write_text(patched)

# Loading the candidate module
spec = importlib.util.spec_from_file_location(
    "dqi_native_candidate", candidate_source
)
dqi_native = importlib.util.module_from_spec(spec)
sys.modules["dqi_native_candidate"] = dqi_native
spec.loader.exec_module(dqi_native)

print("Candidate module loaded:", candidate_source)
print("Builder signature:", inspect.signature(dqi_native.build_reference_circuit))

Candidate module loaded: /content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/dqi_reference_native_candidate.py
Builder signature: (m: 'int', n: 'int', degree: 'int') -> 'km.Circuit'


In [ ]:
test_candidate = dqi_native.build_reference_circuit(
    m=7,
    n=4,
    degree=3
)

s = test_candidate.operation_statistics()

print("Candidate built")
print("Qubits:", test_candidate.num_qubits)
print("Instructions:", s.num_instructions)
print("Toffoli-like:", s.num_toffoli_like)

Candidate built
Qubits: 100
Instructions: 147142
Toffoli-like: 19768


In [ ]:
dqi_candidate = dqi_native.build_reference_circuit(
    m=4095,
    n=70,
    degree=12
)

dqi_candidate_path = (
    COMP_ROOT / "Experiments/DQI/circuits/dqi_native_threshold_v1.kmb"
)

dqi_candidate_path.parent.mkdir(parents=True, exist_ok=True)
dqi_candidate.to_file(str(dqi_candidate_path))

s = dqi_candidate.operation_statistics()

print("Saved:", dqi_candidate_path)
print("Qubits:", dqi_candidate.num_qubits)
print("Instructions:", s.num_instructions)
print("Toffoli-like:", s.num_toffoli_like)

Saved: /content/drive/MyDrive/Quantum_Competition/Experiments/DQI/circuits/dqi_native_threshold_v1.kmb
Qubits: 1893
Instructions: 86233498
Toffoli-like: 5750769


In [ ]:
print("\n".join(patched.splitlines()[312:330]))
print("\n".join(patched.splitlines()[392:415]))

                builder.del_gf2_div_with_scaffold(
                    reg_b[index], reg_a[index], target=ratio, scaffold=scaffold, field=gf)
                builder.free(scaffold + ratio)

        def sweep_positions() -> None:
            with builder.mark('sweep_positions'):
                lim_q = min(m - 1, 1 + it)
                lim_q1 = min(m, 1 + it)
                lim_a = min(m - 2, it // 2 + 1)
                lim_A = m - 2
                order = range(m) if is_adjoint else range(m - 1, -1, -1)

                with (
                    builder.unary_iteration(n_q, control=c0_hc) as eq_i,
                    builder.unary_iteration(n_q, control=c3_hc) as eq_ip1,
                    builder.unary_iteration(n_a, control=c3_hc) as cur_a,
                    builder.unary_iteration(n_A, control=c0_hc) as cur_A,
                ):
            lim_B = m - 2
            lim_b = min(m - 2, it // 2 + 1)

            builder.x(flag)
            nf = builder.alloc_qubits(2)
        

In [ ]:
v2 = patched

# Removing unused unary cursor contexts
for line in [
    "                    builder.unary_iteration(n_a, control=c3_hc) as cur_a,\n",
    "                    builder.unary_iteration(n_A, control=c0_hc) as cur_A,\n",
    "                builder.unary_iteration(n_B, control=ctrl_nB) as cur_B_ge,\n",
    "                builder.unary_iteration(n_b, control=c2_hc) as cur_b_ge,\n",
    "                builder.unary_iteration(n_B, control=c3_nf) as cur_B_lt,\n",
]:
    v2 = v2.replace(line, "")

# Removing cursor arguments from threshold calls
for cursor_name, address_name in [
    ("cur_a", "n_a"),
    ("cur_A", "n_A"),
    ("cur_B_ge", "n_B"),
    ("cur_b_ge", "n_b"),
    ("cur_B_lt", "n_B"),
]:
    v2 = v2.replace(
        f"_xor_threshold(builder, {address_name}, {cursor_name},",
        f"_xor_threshold(builder, {address_name},"
    )

# Updating helper signature
v2 = v2.replace(
    "def _xor_threshold(builder, address, cursor, limit, target, control,",
    "def _xor_threshold(builder, address, limit, target, control,"
)

v2_path = DF_EXP / "dqi_reference_native_candidate_v2.py"
v2_path.write_text(v2)

spec = importlib.util.spec_from_file_location("dqi_native_v2", v2_path)
dqi_native_v2 = importlib.util.module_from_spec(spec)
sys.modules["dqi_native_v2"] = dqi_native_v2
spec.loader.exec_module(dqi_native_v2)

print("V2 loaded:", v2_path)

V2 loaded: /content/drive/MyDrive/Quantum_Competition/Experiments/DFTHCSOS/dqi_reference_native_candidate_v2.py


In [ ]:
p = v2_path
src = p.read_text()
lines = src.splitlines()

for i in range(395, 415):
    print(f"{i+1}: {lines[i]}")

396:             builder.x(flag)
397:             nf = builder.alloc_qubits(2)
398:             builder.init_and(km.array([c0_hc, c3_hc]), flag, nf)
399:             builder.x(flag)
400:             c0_nf, c3_nf = nf
401: 
402:             ctrl_nB = builder.alloc_qubits(1)[0]
403:             builder.cx(km.array([c0_nf, c1_hc]), ctrl_nB)
404: 
405:             order = range(1, m) if is_adjoint else range(m - 1, 0, -1)
406:             with (
407:             ):
408:                 do_shift = builder.alloc_qubits(1)[0]
409: 
410:                 def endpoint(i: int) -> None:
411:                     _xor_threshold(builder, n_B, lim_B, do_shift, ctrl_nB, m - i + 1,
412:                                    at_least=True)
413:                     _xor_threshold(builder, n_b, lim_b, do_shift, c2_hc, i,
414:                                    at_least=True)
415:                     _xor_threshold(builder, n_B, lim_B, do_shift, c3_nf, m - i,


In [ ]:
src = v2_path.read_text()

src = src.replace(
    "            with builder.unary_iteration(n_B, control=ctrl_nB) as cur_B_ge:\n",
    "            with (\n"
    "                builder.unary_iteration(n_B, control=ctrl_nB) as cur_B_ge,\n"
    "                builder.unary_iteration(n_b, control=c2_hc) as cur_b_ge,\n"
    "                builder.unary_iteration(n_B, control=c3_nf) as cur_B_lt,\n"
    "            ):\n"
)

v2_path.write_text(src)

spec = importlib.util.spec_from_file_location("dqi_native_v2", v2_path)
dqi_native_v2 = importlib.util.module_from_spec(spec)
sys.modules["dqi_native_v2"] = dqi_native_v2
spec.loader.exec_module(dqi_native_v2)

test_v2 = dqi_native_v2.build_reference_circuit(m=7, n=4, degree=3)
s = test_v2.operation_statistics()
print("V2:", test_v2.num_qubits, s.num_instructions, s.num_toffoli_like)

V2: 100 147142 19768


In [ ]:
print([x for x in dir(km) if "write" in x.lower() or "save" in x.lower() or "dump" in x.lower()])
print([x for x in dir(full_v2) if "file" in x.lower() or "write" in x.lower() or "serialize" in x.lower()])

[]
['from_file', 'to_file']


In [ ]:
save_path = COMP_ROOT / "Experiments/DQI/circuits/dqi_native_threshold_v2.kmb"
save_path.parent.mkdir(parents=True, exist_ok=True)

full_v2.to_file(str(save_path))

s = full_v2.operation_statistics()
print("Saved:", save_path)
print("Exists:", save_path.exists())
print("Qubits:", full_v2.num_qubits)
print("Instructions:", s.num_instructions)
print("Toffoli-like:", s.num_toffoli_like)

Saved: /content/drive/MyDrive/Quantum_Competition/Experiments/DQI/circuits/dqi_native_threshold_v2.kmb
Exists: True
Qubits: 1893
Instructions: 86233498
Toffoli-like: 5750769


In [ ]:
baseline = dqi.build_reference_circuit(m=4095, n=70, degree=12)
s = baseline.operation_statistics()
print("Baseline:", baseline.num_qubits, s.num_instructions, s.num_toffoli_like)

Baseline: 1893 86051014 5725461


In [ ]:
baseline_path = COMP_ROOT / "Experiments/DQI/circuits/dqi_baseline.kmb"
baseline.to_file(str(baseline_path))

print("Saved:", baseline_path)
print("Size (MB):", round(baseline_path.stat().st_size / 1e6, 2))

Saved: /content/drive/MyDrive/Quantum_Competition/Experiments/DQI/circuits/dqi_baseline.kmb
Size (MB): 1536.8


In [ ]:
result = subprocess.run(
    [
        "python", str(VERIFIER), str(baseline_path),
        "--max-workers", "1",
        "--num-shots", "102400",
        "--batch-size", "256"
    ],
    capture_output=True, text=True,
    env={**os.environ, "PYTHONPATH": str(ROOT) + ":" + str(KICKMIX)}
)

print("Return code:", result.returncode)
print(result.stdout[-2500:])
print(result.stderr[-1000:])

Return code: -9




In [ ]:
FINAL = COMP_ROOT / "Final_Submission"
required = [
    FINAL / "invsqrt.kmb",
    FINAL / "dfthcsos" / "submission.npz",
    FINAL / "dqi.kmb"
]

for p in required:
    print(("FOUND" if p.exists() else "MISSING"), p,
          f"{p.stat().st_size / 1e6:.2f} MB" if p.exists() else "")

FOUND /content/drive/MyDrive/Quantum_Competition/Final_Submission/invsqrt.kmb 0.56 MB
FOUND /content/drive/MyDrive/Quantum_Competition/Final_Submission/dfthcsos/submission.npz 4.54 MB
MISSING /content/drive/MyDrive/Quantum_Competition/Final_Submission/dqi.kmb 


In [ ]:
dqi_final = FINAL / "dqi.kmb"
shutil.copy2(baseline_path, dqi_final)

print("DQI copied:", dqi_final.exists())
print("Size (MB):", round(dqi_final.stat().st_size / 1e6, 2))

print("\nDFTHCSOS files:")
for p in sorted((FINAL / "dfthcsos").iterdir()):
    print(p.name, round(p.stat().st_size / 1e6, 2), "MB")

DQI copied: True
Size (MB): 1536.8

DFTHCSOS files:
angle_k_0.kmb 1.67 MB
angle_k_1.kmb 1.66 MB
angle_k_2.kmb 1.67 MB
angle_k_3.kmb 1.67 MB
angle_k_4.kmb 1.95 MB
angle_k_5.kmb 1.95 MB
angle_k_6.kmb 1.95 MB
angle_k_7.kmb 1.97 MB
angle_k_8.kmb 1.96 MB
angle_k_9.kmb 1.95 MB
submission.npz 4.54 MB


In [ ]:
ZIP_PATH = COMP_ROOT / "Quantum_Competition_Final.zip"

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    # Final submission
    for p in FINAL.rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(COMP_ROOT))

    # Experiment logs and records
    for folder in [
        COMP_ROOT / "Experiments" / "INVSQRT",
        COMP_ROOT / "Experiments" / "DQI",
        COMP_ROOT / "Experiments" / "DFTHCSOS"
    ]:
        for p in folder.rglob("*"):
            if p.is_file() and p.suffix.lower() != ".kmb":
                z.write(p, p.relative_to(COMP_ROOT))

print("ZIP created:", ZIP_PATH.exists())
print("ZIP size (GB):", round(ZIP_PATH.stat().st_size / 1e9, 2))
print("Files:", len(zipfile.ZipFile(ZIP_PATH).namelist()))

ZIP created: True
ZIP size (GB): 0.19
Files: 30


In [ ]:
from google.colab import files
files.download(str(ZIP_PATH))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
p = ROOT / "sample_submission" / "dqi.kmb"

print("Original sample exists:", p.exists())
print("Original size:", round(p.stat().st_size / 1e6, 2), "MB")

with open(p, "rb") as f:
    print("Original magic bytes:", f.read(16))

with open(baseline_path, "rb") as f:
    print("Saved baseline magic:", f.read(16))

Original sample exists: True
Original size: 886.12 MB
Original magic bytes: b'\xd7P\xc7\xd5\xc3)\xd3&\xe3\xcc\x9fh4\xf2\xb8\xbf'
Saved baseline magic: b'APPEND_TO_REGIST'


In [ ]:
import shutil

original_dqi = ROOT / "sample_submission" / "dqi.kmb"
dqi_final = FINAL / "dqi.kmb"

shutil.copy2(original_dqi, dqi_final)

with open(dqi_final, "rb") as f:
    magic = f.read(16)

print("Copied:", dqi_final.exists())
print("Magic bytes:", magic)
print("Size (MB):", round(dqi_final.stat().st_size / 1e6, 2))

Copied: True
Magic bytes: b'\xd7P\xc7\xd5\xc3)\xd3&\xe3\xcc\x9fh4\xf2\xb8\xbf'
Size (MB): 886.12


In [ ]:
ZIP_PATH = COMP_ROOT / "submission.zip"

with zipfile.ZipFile(
    ZIP_PATH, "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=1
) as z:
    for p in FINAL.rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(FINAL))

with zipfile.ZipFile(ZIP_PATH) as z:
    names = z.namelist()
    print("ZIP:", ZIP_PATH)
    print("Files:", len(names))
    print("DQI at root:", "dqi.kmb" in names)
    print("INVSQRT at root:", "invsqrt.kmb" in names)
    print("DFTHCSOS:", sum(x.startswith("dfthcsos/") for x in names))
    print("ZIP size (MB):", round(ZIP_PATH.stat().st_size / 1e6, 1))

ZIP: /content/drive/MyDrive/Quantum_Competition/submission.zip
Files: 13
DQI at root: True
INVSQRT at root: True
DFTHCSOS: 11
ZIP size (MB): 122.9


In [ ]:
from google.colab import files
files.download(str(ZIP_PATH))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>